# D-24: Notebook 4 — Layer Ablation Study

> **Project:** FractalRecall — Hierarchical context-aware embedding retrieval
> **Question this notebook answers:** Which individual context layers contribute most to retrieval improvement?
> **Corpus:** Aethelgard worldbuilding (Markdown + YAML frontmatter)
> **Embedding model:** `nomic-embed-text-v1.5` via sentence-transformers
> **Vector DB:** ChromaDB (in-memory)
> **Metrics:** Precision@5, Recall@10, NDCG@10, MRR
> **Full context for AI assistants:** See `COLAB-SESSION-CONTEXT.md` in this directory

## What is FractalRecall?

FractalRecall tests the hypothesis that embedding retrieval improves when text chunks are enriched with **hierarchical structural context** before embedding. Each chunk carries up to 8 "context layers" prepended as natural language prefixes. The enriched text is embedded as a single vector.

## Notebook Sequence

| # | Notebook | Status |
|---|----------|--------|
| 1 | Baseline (standard RAG, no enrichment) | ✅ D-21 |
| 2 | Single-Layer Enrichment (document-level context only) | ✅ D-22 |
| 3 | Multi-Layer Enrichment (**core hypothesis test**) | ✅ D-23 (4 rounds) |
| 4 | **Layer Ablation (which layers matter most?)** | ✅ D-24 ← THIS NOTEBOOK |
| 5 | Embedding Strategy Comparison | 🔲 |
| 6 | Cross-Domain Validation | 🔲 |

## D-24 Ablation Design

### Motivation

D-23 showed that **6-layer enrichment (~80 tokens) degrades retrieval breadth** (P@5, R@10)
compared to both D-21 (no enrichment) and D-22 (single-layer, ~24 tokens), while improving
first-result ranking (MRR). The hypothesis is that fewer, more targeted layers can retain
the MRR benefit without the content-dilution penalty.

### Ablation Configurations

| Config | Label | Layers | ~Prefix Tokens | Purpose |
|--------|-------|--------|----------------|---------|
| **A** | `raw` | None | 0 | Control (D-21 equivalent) |
| **B** | `domain_entity` | Domain + Entity | ~32 | Minimal enrichment |
| **C** | `de_authority` | Domain + Entity + Authority | ~42 | +canonical status |
| **D** | `de_section` | Domain + Entity + Section | ~48 | +document structure |
| **E** | `de_relationships` | Domain + Entity + Relationships | ~50 | +cross-references |

### Dropped Layers

- **Corpus**: Constant string identical for all chunks — zero discriminative value (6 tokens wasted)
- **Temporal**: 0% population across D-23 rounds 1-4 — corpus lacks temporal_markers frontmatter

### Chunking

All configs use D-21's exact chunking algorithm (ported in D-23 R4):
- Config A: `prefix_reserve=0` (matches D-21 geometry exactly)
- Configs B-E: `prefix_reserve=100` (same as D-23 R4)

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 02: Install Dependencies

Installs all required packages for D-23 notebook execution.
Same package set as D-21 and D-22 for consistency.

Packages:
  - chromadb: Vector database for embedding storage and retrieval
  - sentence-transformers: Embedding model framework (Nomic v2-moe, v1.5)
  - FlagEmbedding: Embedding model framework (BAAI BGE-M3)
  - transformers: Hugging Face model infrastructure
  - torch: PyTorch backend for embedding computation
  - numpy, pandas: Numerical/data analysis
  - scipy: Statistical testing (Wilcoxon signed-rank)
  - scikit-learn: ML utilities
  - matplotlib, seaborn: Visualization
  - tqdm: Progress bars
  - pyyaml: YAML frontmatter parsing
  - umap-learn: Dimensionality reduction for embedding visualization
"""

# Install required packages (suppress verbose output with -q)
!pip install -q chromadb sentence-transformers FlagEmbedding transformers torch numpy pandas scipy scikit-learn matplotlib seaborn tqdm pyyaml umap-learn

# ============================================================================
# VERSION VERIFICATION
# ============================================================================

print("✓ Dependency Installation Complete\n")
print("Package Version Check:")
print("-" * 50)

packages_to_check = [
    'chromadb',
    'sentence_transformers',
    'torch',
    'numpy',
    'pandas',
    'scipy',
    'sklearn',
    'matplotlib',
    'seaborn',
    'tqdm',
    'yaml',
]

for package in packages_to_check:
    try:
        mod = __import__(package)
        version = getattr(mod, '__version__', 'installed (no version attr)')
        print(f"  {package:25} {version}")
    except ImportError:
        print(f"  {package:25} [IMPORT FAILED]")

print("-" * 50)
print("\n✓ All critical dependencies installed and verified.")

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 03: Imports, Configuration, Model Selection

Defines all imports, model configurations, and path constants for D-23.
Extends D-22 config with multi-layer enrichment parameters.

Key D-23 additions vs D-22:
  - prefix_reserve_tokens increased to 100-150 (from D-22's ~30) for 8-layer prefix
  - BONFERRONI_PAIRS = 3 for 3-way statistical correction
  - ADJUSTED_ALPHA = 0.05 / 3 ≈ 0.0167
  - CORPUS_LABEL constant for Corpus layer
  - Output paths for D-23 specific artifacts
"""

# ============================================================================
# STANDARD LIBRARY IMPORTS
# ============================================================================
import os
import sys
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any, Optional
import yaml
import re
import json
from datetime import datetime

# ============================================================================
# THIRD-PARTY IMPORTS
# ============================================================================
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings

import chromadb
from chromadb.config import Settings

warnings.filterwarnings('ignore')

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================

# ── USER MUST UPDATE THIS AFTER RUNNING D-21 ──
# Default: "v1.5" — the expected winner based on 8192-token context window
# and strong general-purpose performance. Update to match D-21's actual winner.
SELECTED_MODEL = "v1.5"  # Options: "v2-moe", "v1.5", "bge-m3"

@dataclass
class ModelConfig:
    """Configuration for an embedding model.

    Attributes:
        name: Short identifier (e.g., "v1.5")
        hf_model_id: Hugging Face model path
        max_tokens: Model's maximum context window in tokens
        max_chunk_tokens: Target maximum chunk size (content + prefix)
        prefix_reserve_tokens: Tokens reserved for enrichment prefix
        embedding_dim: Output embedding dimensionality
        task_prefix_doc: Prefix for document indexing (empty for BGE-M3)
        task_prefix_query: Prefix for query encoding (empty for BGE-M3)
    """
    name: str
    hf_model_id: str
    max_tokens: int
    max_chunk_tokens: int
    prefix_reserve_tokens: int
    embedding_dim: int
    task_prefix_doc: str
    task_prefix_query: str

# Three models supported — same as D-21/D-22
MODELS: Dict[str, ModelConfig] = {
    "v2-moe": ModelConfig(
        name="v2-moe",
        hf_model_id="nomic-ai/nomic-embed-text-v2-moe",
        max_tokens=512,
        max_chunk_tokens=350,
        prefix_reserve_tokens=100,   # 8-layer prefix: ~80-150 tokens
        embedding_dim=768,
        task_prefix_doc="search_document: ",
        task_prefix_query="search_query: ",
    ),
    "v1.5": ModelConfig(
        name="v1.5",
        hf_model_id="nomic-ai/nomic-embed-text-v1.5",
        max_tokens=8192,
        max_chunk_tokens=1024,       # [FIX Round 3] Match D-21/D-22 for controlled comparison
        prefix_reserve_tokens=100,   # [FIX Round 3] Actual max overhead is 84 tokens; 100 is safe
        embedding_dim=768,
        task_prefix_doc="search_document: ",
        task_prefix_query="search_query: ",
    ),
    "bge-m3": ModelConfig(
        name="bge-m3",
        hf_model_id="BAAI/bge-m3",
        max_tokens=8192,
        max_chunk_tokens=1024,
        prefix_reserve_tokens=150,   # 8-layer prefix: ~80-150 tokens
        embedding_dim=1024,
        task_prefix_doc="",          # BGE-M3 doesn't use task prefixes
        task_prefix_query="",
    ),
}

# Validate selection
if SELECTED_MODEL not in MODELS:
    raise ValueError(
        f"SELECTED_MODEL '{SELECTED_MODEL}' not in {list(MODELS.keys())}. "
        f"Update after running D-21."
    )

MODEL_CONFIG = MODELS[SELECTED_MODEL]

# ============================================================================
# PATH CONSTANTS
# ============================================================================

# Corpus directory (D-20 output)
CORPUS_DIR = Path("./corpus")

# D-23 output directory
OUTPUT_DIR = Path("./d23-output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ChromaDB persistence directory
CHROMADB_DIR = OUTPUT_DIR / "chromadb"
CHROMADB_DIR.mkdir(parents=True, exist_ok=True)

# Prior notebook results for 3-way comparison
D21_RESULTS_PATH = Path("./d21-output/d21_results.csv")
D22_RESULTS_PATH = Path("./d22-output/d22_results.csv")

# ============================================================================
# EVALUATION CONSTANTS
# ============================================================================

K_PRECISION = 5      # Precision@5
K_RECALL = 10        # Recall@10
K_NDCG = 10          # NDCG@10

# Metric column names used throughout the notebook
METRICS = ["precision@5", "recall@10", "ndcg@10", "mrr"]
METRIC_LABELS = ["P@5", "R@10", "NDCG@10", "MRR"]

# ============================================================================
# STATISTICAL TESTING CONSTANTS
# ============================================================================

SIGNIFICANCE_LEVEL = 0.05

# 3-way comparison requires Bonferroni correction:
# 3 pairwise comparisons: D-23 vs D-21, D-23 vs D-22, D-22 vs D-21
BONFERRONI_PAIRS = 3
ADJUSTED_ALPHA = SIGNIFICANCE_LEVEL / BONFERRONI_PAIRS  # ≈ 0.0167

# ============================================================================
# MULTI-LAYER ENRICHMENT CONSTANTS
# ============================================================================

# Corpus layer label (hardcoded for this corpus)
CORPUS_LABEL = "Aethelgard Worldbuilding Corpus v5.0"

# ============================================================================
# PRINT CONFIGURATION SUMMARY
# ============================================================================

print("✓ Imports and Configuration Complete\n")
print("=" * 70)
print("D-23 CONFIGURATION SUMMARY")
print("=" * 70)

print(f"\nModel Selection:")
print(f"  SELECTED_MODEL:       {SELECTED_MODEL}")
print(f"  Model ID:             {MODEL_CONFIG.hf_model_id}")
print(f"  Max tokens:           {MODEL_CONFIG.max_tokens}")
print(f"  Max chunk tokens:     {MODEL_CONFIG.max_chunk_tokens}")
print(f"  Prefix reserve:       {MODEL_CONFIG.prefix_reserve_tokens} tokens (multi-layer)")
print(f"  Effective content:    {MODEL_CONFIG.max_chunk_tokens - MODEL_CONFIG.prefix_reserve_tokens} tokens")
print(f"  Embedding dimension:  {MODEL_CONFIG.embedding_dim}")

print(f"\nEvaluation:")
print(f"  Metrics:              {', '.join(METRICS)}")
print(f"  Alpha (unadjusted):   {SIGNIFICANCE_LEVEL}")
print(f"  Bonferroni pairs:     {BONFERRONI_PAIRS}")
print(f"  Adjusted alpha:       {ADJUSTED_ALPHA:.4f}")

print(f"\nPaths:")
print(f"  Corpus:               {CORPUS_DIR}")
print(f"  Output:               {OUTPUT_DIR}")
print(f"  ChromaDB:             {CHROMADB_DIR}")
print(f"  D-21 results:         {D21_RESULTS_PATH}")
print(f"  D-22 results:         {D22_RESULTS_PATH}")

print(f"\nEnrichment:")
print(f"  Corpus label:         {CORPUS_LABEL}")
print(f"  Layers:               8 (Corpus, Domain, Entity, Authority, Temporal, Relational, Section, Content)")

print("=" * 70)
print("\n✓ Configuration complete. Ready for corpus loading.")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 04: Ground-Truth Query Set (Aligned with D-21/D-22)

Defines 36 ground-truth queries for evaluation, organized by type:
  - SINGLE_HOP (11): Direct attribute lookups
  - MULTI_HOP (8): Cross-entity relationship traversal
  - AUTHORITY (5): Canonical vs. draft vs. superseded status questions
  - TEMPORAL (6): Time-based historical queries
  - EXPLORATORY (6): Open-ended relationship discovery

Same query set as D-21 Cell 05 and D-22 Cell 04 for consistent 3-way comparison.

Each query includes:
  - query_id: Unique identifier (e.g., "Q-01")
  - type: Category (SINGLE_HOP, MULTI_HOP, AUTHORITY, TEMPORAL, EXPLORATORY)
  - text: Natural language query string
  - relevant_docs: List of corpus files expected to be relevant
"""

@dataclass
class GroundTruthQuery:
    """A single ground-truth query with expected relevant documents.

    Attributes:
        query_id: Unique query identifier (e.g., "Q-01")
        query_text: Natural language query
        query_type: One of SINGLE_HOP, MULTI_HOP, AUTHORITY, TEMPORAL, EXPLORATORY
        expected_filenames: List of corpus filenames expected to be relevant
        relevance_scores: Dict mapping filename → relevance grade (1-3)
    """
    query_id: str
    query_text: str
    query_type: str
    expected_filenames: List[str]
    relevance_scores: Dict[str, int]

# ============================================================================
# FULL QUERY SET: 36 queries (identical to D-21/D-22)
# ============================================================================

# SINGLE_HOP QUERIES (11 total)
# These test direct lookups of facts, entities, and concepts

QUERIES = [
    {
        "query_id": "Q-01",
        "type": "SINGLE_HOP",
        "text": "What is the Echo-Cant communication system?",
        "relevant_docs": ["000-codex_echo-cant.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-02",
        "type": "SINGLE_HOP",
        "text": "What are the defining characteristics of the Void-Marked?",
        "relevant_docs": ["db02-wb_void-marked-assembled-entry.md", "db03-dc_jotun-reader-chronology.md"],
    },
    {
        "query_id": "Q-03",
        "type": "SINGLE_HOP",
        "text": "Describe the Harrow-Sick condition and its effects.",
        "relevant_docs": ["000-codex_harrow-sick.md", "db02-wb_medical-phenomena-entry.md"],
    },
    {
        "query_id": "Q-04",
        "type": "SINGLE_HOP",
        "text": "What is the ODIN Protocol?",
        "relevant_docs": ["000-codex_odin-protocol.md", "standalone_aether-weave-os.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-05",
        "type": "SINGLE_HOP",
        "text": "What is the \u00c6ther-Weave operating system?",
        "relevant_docs": ["standalone_aether-weave-os.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-06",
        "type": "SINGLE_HOP",
        "text": "Who are the Scavenger-Barons and what do they do?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "db03-dc_contract-dispute-case-study.md"],
    },
    {
        "query_id": "Q-07",
        "type": "SINGLE_HOP",
        "text": "What is the Nine-Tiers architectural framework?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-27",
        "type": "SINGLE_HOP",
        "text": "What are the primary functions of the Warden-Host?",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-28",
        "type": "SINGLE_HOP",
        "text": "Define the Glitch in the context of Aethelgard's history.",
        "relevant_docs": ["000-resources_comprehensive-glossary.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-30",
        "type": "SINGLE_HOP",
        "text": "What is the Spell-Lock system?",
        "relevant_docs": ["000-codex_spell-lock.md", "standalone_aether-weave-os.md"],
    },
    {
        "query_id": "Q-34",
        "type": "SINGLE_HOP",
        "text": "Describe the Weir-Bone material and its properties.",
        "relevant_docs": ["db02-wb_weir-bone-assembled-entry.md", "000-resources_comprehensive-glossary.md"],
    },

    # MULTI_HOP QUERIES (8 total)
    # These require traversing relationships between multiple entities/concepts

    {
        "query_id": "Q-08",
        "type": "MULTI_HOP",
        "text": "How do Iron-Bane and God-Sleeper theological positions on Svin-fylking differ?",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md", "db02-wb_svin-fylking-assembled-entry.md"],
    },
    {
        "query_id": "Q-09",
        "type": "MULTI_HOP",
        "text": "What is the relationship between the Harrow-Sick and the Void-Marked?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db02-wb_void-marked-assembled-entry.md", "db03-dc_medical-causality-research.md"],
    },
    {
        "query_id": "Q-10",
        "type": "MULTI_HOP",
        "text": "How do the Scavenger-Barons use Spell-Lock in their operations?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "000-codex_spell-lock.md", "db03-dc_contract-dispute-case-study.md"],
    },
    {
        "query_id": "Q-11",
        "type": "MULTI_HOP",
        "text": "Explain the conflict between Warden-Host sanctuary protocols and Iron-Bane territorial claims.",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db03-dc_iron-bane-theological-analysis.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-12",
        "type": "MULTI_HOP",
        "text": "How does the ODIN Protocol interact with the \u00c6ther-Weave OS?",
        "relevant_docs": ["000-codex_odin-protocol.md", "standalone_aether-weave-os.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-29",
        "type": "MULTI_HOP",
        "text": "What role does the Echo-Cant system play in maintaining Warden-Host operations?",
        "relevant_docs": ["000-codex_echo-cant.md", "db02-wb_warden-host-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-31",
        "type": "MULTI_HOP",
        "text": "How do Void-Marked and Weir-Bone materials interact in salvage contexts?",
        "relevant_docs": ["db02-wb_void-marked-assembled-entry.md", "db02-wb_weir-bone-assembled-entry.md", "db03-dc_salvage-operations-manual.md"],
    },
    {
        "query_id": "Q-35",
        "type": "MULTI_HOP",
        "text": "What is the relationship between the Nine-Tiers architecture and Svin-fylking religious doctrine?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "db02-wb_svin-fylking-assembled-entry.md", "db03-dc_god-sleeper-operational-doctrine.md"],
    },

    # AUTHORITY QUERIES (5 total)
    # These test knowledge of canonical vs. draft vs. superseded information

    {
        "query_id": "Q-13",
        "type": "AUTHORITY",
        "text": "What is the canonical explanation for how the Glitch occurred?",
        "relevant_docs": ["000-resources_comprehensive-glossary.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-14",
        "type": "AUTHORITY",
        "text": "What are the established facts about the God-Sleeper movement origins?",
        "relevant_docs": ["db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-15",
        "type": "AUTHORITY",
        "text": "Which interpretations of Harrow-Sick etiology are considered canon?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db03-dc_medical-causality-research.md"],
    },
    {
        "query_id": "Q-16",
        "type": "AUTHORITY",
        "text": "What is the official Warden-Host stance on Void-Marked rights?",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db02-wb_void-marked-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-17",
        "type": "AUTHORITY",
        "text": "According to published sources, what materials constitute a valid Spell-Lock?",
        "relevant_docs": ["000-codex_spell-lock.md", "000-resources_comprehensive-glossary.md"],
    },

    # TEMPORAL QUERIES (6 total)
    # These test time-based historical retrieval across the Aethelgard timeline

    {
        "query_id": "Q-18",
        "type": "TEMPORAL",
        "text": "What major events happened in the first century after the Glitch (Year 0-100 PG)?",
        "relevant_docs": ["db03-dc_jotun-reader-chronology.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-19",
        "type": "TEMPORAL",
        "text": "When was the Warden-Host sanctuary established and what precipitated it?",
        "relevant_docs": ["db03-dc_sanctuary-establishment-record.md", "db02-wb_warden-host-assembled-entry.md"],
    },
    {
        "query_id": "Q-20",
        "type": "TEMPORAL",
        "text": "Trace the chronological development of Iron-Bane theological doctrine.",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-21",
        "type": "TEMPORAL",
        "text": "What is the timeline of major salvage discoveries in the Aethelgard region?",
        "relevant_docs": ["db03-dc_salvage-operations-manual.md", "db03-dc_jotun-reader-chronology.md"],
    },
    {
        "query_id": "Q-22",
        "type": "TEMPORAL",
        "text": "When did the God-Sleeper movement gain significant political influence?",
        "relevant_docs": ["db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-23",
        "type": "TEMPORAL",
        "text": "Describe the sequence of events in the Scavenger-Baron contract dispute.",
        "relevant_docs": ["db03-dc_contract-dispute-case-study.md", "db02-wb_scavenger-barons-assembled-entry.md"],
    },

    # EXPLORATORY QUERIES (6 total)
    # These test open-ended discovery of related concepts and themes

    {
        "query_id": "Q-24",
        "type": "EXPLORATORY",
        "text": "What are the major political tensions in post-Glitch Aethelgard?",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_contract-dispute-case-study.md", "db02-wb_scavenger-barons-assembled-entry.md"],
    },
    {
        "query_id": "Q-25",
        "type": "EXPLORATORY",
        "text": "How do different factions view the technological salvage efforts?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "db03-dc_salvage-operations-manual.md", "db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md"],
    },
    {
        "query_id": "Q-26",
        "type": "EXPLORATORY",
        "text": "What medical and physiological mysteries remain unsolved in Aethelgard?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db03-dc_medical-causality-research.md", "db02-wb_void-marked-assembled-entry.md"],
    },
    {
        "query_id": "Q-32",
        "type": "EXPLORATORY",
        "text": "What are the intersections between religious doctrine and technological systems in Aethelgard?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "db02-wb_svin-fylking-assembled-entry.md", "000-codex_odin-protocol.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-33",
        "type": "EXPLORATORY",
        "text": "How do material properties (Weir-Bone, Void-Marked) influence cultural practices?",
        "relevant_docs": ["db02-wb_weir-bone-assembled-entry.md", "db02-wb_void-marked-assembled-entry.md", "db02-wb_warden-host-assembled-entry.md"],
    },
    {
        "query_id": "Q-36",
        "type": "EXPLORATORY",
        "text": "What gaps exist in the documented understanding of Aethelgard's pre-Glitch history?",
        "relevant_docs": ["db03-dc_historical-theological-survey.md", "db03-dc_jotun-reader-chronology.md", "000-resources_comprehensive-glossary.md"],
    },
]

# ============================================================================
# BUILD GROUND_TRUTH_QUERIES from QUERIES (for compatibility with Cell 14)
# ============================================================================

GROUND_TRUTH_QUERIES: List[GroundTruthQuery] = []
for q in QUERIES:
    # Build uniform relevance scores (all docs score 2 by default)
    relevance = {doc: 2 for doc in q["relevant_docs"]}
    GROUND_TRUTH_QUERIES.append(GroundTruthQuery(
        query_id=q["query_id"],
        query_text=q["text"],
        query_type=q["type"],
        expected_filenames=q["relevant_docs"],
        relevance_scores=relevance,
    ))

# ============================================================================
# VALIDATION
# ============================================================================

query_type_counts = {}
for q in QUERIES:
    query_type_counts[q["type"]] = query_type_counts.get(q["type"], 0) + 1

assert len(QUERIES) == 36, f"Expected 36 queries, got {len(QUERIES)}"
assert query_type_counts.get("SINGLE_HOP", 0) == 11, f"Expected 11 SINGLE_HOP"
assert query_type_counts.get("MULTI_HOP", 0) == 8, f"Expected 8 MULTI_HOP"
assert query_type_counts.get("AUTHORITY", 0) == 5, f"Expected 5 AUTHORITY"
assert query_type_counts.get("TEMPORAL", 0) == 6, f"Expected 6 TEMPORAL"
assert query_type_counts.get("EXPLORATORY", 0) == 6, f"Expected 6 EXPLORATORY"

print(f"\u2713 All 36 ground-truth queries validated (aligned with D-21/D-22)")
print(f"  Distribution: {query_type_counts}")
print(f"  GROUND_TRUTH_QUERIES: {len(GROUND_TRUTH_QUERIES)} GroundTruthQuery objects")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 05: Field Mapping & Normalization

Normalizes raw YAML frontmatter fields to consistent canonical forms.
Handles variations in field names (e.g., "canon" vs "canonical_status")
and value formats (e.g., True vs "yes" vs "canonical").

Identical to D-21 Cell 05 and D-22 Cell 05 for consistency.

Functions:
  - normalize_canon_status(value) → str ("true"/"false")
  - normalize_authority_layer(value) → str ("primary"/"secondary"/"tertiary"/"unknown")
  - normalize_entity_type(value) → str ("character"/"location"/"faction"/etc.)
  - normalize_version(value) → str (semantic version like "1.0")
  - map_frontmatter(raw) → Dict[str, Any] (fully normalized metadata dict)
"""

# ============================================================================
# FIELD MAP: maps canonical key → list of possible raw frontmatter key names
# ============================================================================

FIELD_MAP: Dict[str, List[str]] = {
    "type":              ["type", "entity_type", "document_type", "doc_type", "category"],
    "name":              ["name", "entity_name", "title", "subject"],
    "canon":             ["canon", "canon_status", "canonical", "is_canonical"],
    "authority_layer":   ["authority_layer", "authority", "authority_level"],
    "era":               ["era", "age", "time_period", "historical_era", "eras", "temporal_markers"],
    "domain_layer":      ["domain_layer", "domain", "lore_category"],
    "related_entities":  ["related_entities", "relationships", "relations", "links", "see_also", "cross_references"],
    "version":           ["version", "doc_version", "revision"],
    "tags":              ["tags", "keywords", "labels"],
    "description":       ["description", "summary", "blurb"],
}


def normalize_canon_status(value: Any) -> str:
    """Normalize canon/canonical status to 'true' or 'false'.

    Handles: True/False booleans, 'yes'/'no', 'canonical'/'apocryphal',
    numeric 1/0, and string variations.

    Args:
        value: Raw canon value from frontmatter

    Returns:
        Normalized string: 'true' or 'false'
    """
    if value is None:
        return "false"
    if isinstance(value, bool):
        return "true" if value else "false"
    s = str(value).lower().strip()
    if s in ("true", "yes", "1", "canonical", "canon"):
        return "true"
    return "false"


def normalize_authority_layer(value: Any) -> str:
    """Normalize authority layer to canonical form.

    Maps various authority designations to: 'primary', 'secondary',
    'tertiary', or 'unknown'.

    Args:
        value: Raw authority value

    Returns:
        One of: 'primary', 'secondary', 'tertiary', 'unknown'
    """
    if value is None:
        return "unknown"
    s = str(value).lower().strip()
    if s in ("primary", "official", "canonical", "1"):
        return "primary"
    if s in ("secondary", "semi-official", "semi-canon", "2"):
        return "secondary"
    if s in ("tertiary", "unofficial", "fan", "apocryphal", "3"):
        return "tertiary"
    return "unknown"


def normalize_entity_type(value: Any) -> str:
    """Normalize entity type to canonical form.

    Maps variations like 'char', 'npc', 'org' to standard names.

    Args:
        value: Raw entity type value

    Returns:
        Normalized type string (e.g., 'character', 'faction', 'location')
    """
    if value is None:
        return "unknown"
    s = str(value).lower().strip()
    type_map = {
        "character": "character", "char": "character", "npc": "character", "person": "character",
        "faction": "faction", "org": "faction", "organization": "faction", "group": "faction",
        "location": "location", "place": "location", "region": "location", "area": "location",
        "event": "event", "battle": "event", "war": "event",
        "item": "item", "artifact": "item", "object": "item", "weapon": "item",
        "concept": "concept", "magic": "concept", "system": "concept",
        "spell": "spell", "ability": "spell",
        "creature": "creature", "monster": "creature", "beast": "creature",
        "timeline": "timeline", "chronology": "timeline",
    }
    return type_map.get(s, s)  # Return as-is if not in map


def normalize_version(value: Any) -> str:
    """Extract semantic version from raw version string.

    Args:
        value: Raw version (e.g., 'v1.2', '2.0.1', 'rev3')

    Returns:
        Semantic version string (e.g., '1.2', '2.0.1'). Defaults to '1.0'.
    """
    if value is None:
        return "1.0"
    match = re.search(r'v?(\d+(?:\.\d+)*)', str(value))
    return match.group(1) if match else "1.0"


def map_frontmatter(raw_frontmatter: Dict[str, Any]) -> Dict[str, Any]:
    """Map raw frontmatter keys to normalized canonical form.

    Iterates through FIELD_MAP to find matching keys in raw_frontmatter,
    applies appropriate normalization, and returns a clean metadata dict.

    Args:
        raw_frontmatter: Dict with potentially inconsistent keys

    Returns:
        Dict with normalized keys and values. Unmapped keys preserved
        with 'raw_' prefix.
    """
    normalized: Dict[str, Any] = {}
    mapped_raw_keys: set = set()

    for canonical_key, variations in FIELD_MAP.items():
        for variation in variations:
            if variation in raw_frontmatter:
                raw_value = raw_frontmatter[variation]
                mapped_raw_keys.add(variation)

                # Apply appropriate normalization
                if canonical_key == "canon":
                    normalized[canonical_key] = normalize_canon_status(raw_value)
                elif canonical_key == "authority_layer":
                    normalized[canonical_key] = normalize_authority_layer(raw_value)
                elif canonical_key == "type":
                    normalized[canonical_key] = normalize_entity_type(raw_value)
                elif canonical_key == "version":
                    normalized[canonical_key] = normalize_version(raw_value)
                else:
                    normalized[canonical_key] = raw_value

                break  # Use first matching variation

    # Preserve unmapped keys with 'raw_' prefix for diagnostics
    for key, value in raw_frontmatter.items():
        if key not in mapped_raw_keys:
            normalized[f"raw_{key}"] = value

    return normalized


# ============================================================================
# VERIFICATION
# ============================================================================

print("✓ Field Mapping & Normalization Functions Loaded\n")
print(f"  Canonical fields: {len(FIELD_MAP)}")
for key, variations in FIELD_MAP.items():
    print(f"    {key:20s} ← {variations}")

print(f"\n  Normalization functions: canon_status, authority_layer, entity_type, version")
print("✓ Ready to normalize corpus frontmatter.")

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 06: Corpus Loading

Loads all 25 lore documents from the D-20 test corpus directory.
Parses YAML frontmatter and Markdown body from each file.
Validates loaded documents against ground-truth query expectations.

Identical to D-21 Cell 06 and D-22 Cell 06 for consistency.

Outputs:
  - corpus: List[LoreDocument] (all loaded documents)
  - filename_index: Dict[str, LoreDocument] (fast lookup by filename)
  - Validation report: which ground-truth expected files are present/missing
"""

from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional
from pathlib import Path
from tqdm import tqdm
import yaml # Added missing import
import re # Added missing import for regex in normalization

# Path constant needed for corpus loading
CORPUS_DIR = Path("./corpus")

# ============================================================================
# FIELD MAP: maps canonical key → list of possible raw frontmatter key names
# (Moved from D-23 Cell 05)
# ============================================================================

FIELD_MAP: Dict[str, List[str]] = {
    "type":              ["type", "entity_type", "document_type", "doc_type", "category"],
    "name":              ["name", "entity_name", "title", "subject"],
    "canon":             ["canon", "canon_status", "canonical", "is_canonical"],
    "authority_layer":   ["authority_layer", "authority", "authority_level"],
    "era":               ["era", "age", "time_period", "historical_era", "eras"],
    "domain_layer":      ["domain_layer", "domain", "lore_category"],
    "related_entities":  ["related_entities", "relationships", "relations", "links", "see_also"],
    "version":           ["version", "doc_version", "revision"],
    "tags":              ["tags", "keywords", "labels"],
    "description":       ["description", "summary", "blurb"],
}

def normalize_canon_status(value: Any) -> str:
    """Normalize canon/canonical status to 'true' or 'false'.
    (Moved from D-23 Cell 05)"""
    if value is None:
        return "false"
    if isinstance(value, bool):
        return "true" if value else "false"
    s = str(value).lower().strip()
    if s in ("true", "yes", "1", "canonical", "canon"):
        return "true"
    return "false"

def normalize_authority_layer(value: Any) -> str:
    """Normalize authority layer to canonical form.
    (Moved from D-23 Cell 05)"""
    if value is None:
        return "unknown"
    s = str(value).lower().strip()
    if s in ("primary", "official", "canonical", "1"):
        return "primary"
    if s in ("secondary", "semi-official", "semi-canon", "2"):
        return "secondary"
    if s in ("tertiary", "unofficial", "fan", "apocryphal", "3"):
        return "tertiary"
    return "unknown"

def normalize_entity_type(value: Any) -> str:
    """Normalize entity type to canonical form.
    (Moved from D-23 Cell 05)"""
    if value is None:
        return "unknown"
    s = str(value).lower().strip()
    type_map = {
        "character": "character", "char": "character", "npc": "character", "person": "character",
        "faction": "faction", "org": "faction", "organization": "faction", "group": "faction",
        "location": "location", "place": "location", "region": "location", "area": "location",
        "event": "event", "battle": "event", "war": "event",
        "item": "item", "artifact": "item", "object": "item", "weapon": "item",
        "concept": "concept", "magic": "concept", "system": "concept",
        "spell": "spell", "ability": "spell",
        "creature": "creature", "monster": "creature", "beast": "creature",
        "timeline": "timeline", "chronology": "timeline",
    }
    return type_map.get(s, s)

def normalize_version(value: Any) -> str:
    """Extract semantic version from raw version string.
    (Moved from D-23 Cell 05)"""
    if value is None:
        return "1.0"
    match = re.search(r'v?(\d+(?:\.\d+)*)', str(value))
    return match.group(1) if match else "1.0"

def map_frontmatter(raw_frontmatter: Dict[str, Any]) -> Dict[str, Any]:
    """Map raw frontmatter keys to normalized canonical form.
    (Moved from D-23 Cell 05)"""
    normalized: Dict[str, Any] = {}
    mapped_raw_keys: set = set()

    for canonical_key, variations in FIELD_MAP.items():
        for variation in variations:
            if variation in raw_frontmatter:
                raw_value = raw_frontmatter[variation]
                mapped_raw_keys.add(variation)

                if canonical_key == "canon":
                    normalized[canonical_key] = normalize_canon_status(raw_value)
                elif canonical_key == "authority_layer":
                    normalized[canonical_key] = normalize_authority_layer(raw_value)
                elif canonical_key == "type":
                    normalized[canonical_key] = normalize_entity_type(raw_value)
                elif canonical_key == "version":
                    normalized[canonical_key] = normalize_version(raw_value)
                else:
                    normalized[canonical_key] = raw_value

                break

    for key, value in raw_frontmatter.items():
        if key not in mapped_raw_keys:
            normalized[f"raw_{key}"] = value

    return normalized

@dataclass
class LoreDocument:
    """A single document from the FractalRecall corpus.

    Attributes:
        filename: Filename (e.g., 'iron_covenant.md')
        frontmatter: Raw YAML frontmatter dict (before normalization)
        body: Markdown body text (after frontmatter)
        metadata: Normalized metadata dict (output of map_frontmatter)
    """
    filename: str
    frontmatter: Dict[str, Any]
    body: str
    metadata: Dict[str, Any]


def parse_lore_file(filepath: Path) -> Tuple[Dict[str, Any], str]:
    """Parse a lore document into frontmatter and body.

    Expects format:
        ---
        key: value
        ---
        # Markdown body

    Args:
        filepath: Path to .md file

    Returns:
        Tuple of (frontmatter_dict, body_str)
    """
    try:
        content = filepath.read_text(encoding="utf-8")
    except Exception as e:
        print(f"  ✗ Error reading {filepath.name}: {e}")
        return {}, ""

    # Split on YAML frontmatter delimiters
    if content.startswith("---"):
        parts = content.split("---", 2)
        if len(parts) >= 3:
            try:
                frontmatter = yaml.safe_load(parts[1]) or {}
            except yaml.YAMLError as e:
                print(f"  ☢ YAML parse error in {filepath.name}: {e}")
                frontmatter = {}
            body = parts[2].strip()
        else:
            frontmatter = {}
            body = content
    else:
        frontmatter = {}
        body = content

    if not isinstance(frontmatter, dict):
        frontmatter = {}

    return frontmatter, body


def estimate_tokens(text: str) -> int:
    """Estimate token count using word-count heuristic.

    Heuristic: tokens ≈ word_count × 1.3
    This is a rough approximation; actual count varies by tokenizer.
    Consistent with D-21/D-22 for fair comparison.

    Args:
        text: Input text string

    Returns:
        Approximate token count (minimum 1)
    """
    if not text or not text.strip():
        return 0
    word_count = len(text.split())
    return max(1, int(word_count * 1.3))

def load_corpus(corpus_dir: Path) -> List[LoreDocument]:
    """Load all .md and .yaml files from corpus directory.

    Args:
        corpus_dir: Path to D-20 test corpus

    Returns:
        List of LoreDocument objects, sorted by filename
    """
    if not corpus_dir.exists():
        print(f"✗ Corpus directory not found: {corpus_dir}")
        print(f"  Ensure D-20 test corpus is available at this path.")
        return []

    # Collect all candidate files
    files = sorted(corpus_dir.glob("*.md"))

    print(f"Loading {len(files)} files from {corpus_dir}...")
    documents = []

    for filepath in tqdm(files, desc="Parsing documents"):
        frontmatter, body = parse_lore_file(filepath)
        metadata = map_frontmatter(frontmatter)

        # Add derived fields
        metadata["filepath"] = str(filepath)
        metadata["filename"] = filepath.name

        doc = LoreDocument(
            filename=filepath.name,
            frontmatter=frontmatter,
            body=body,
            metadata=metadata,
        )
        documents.append(doc)

    return documents


# ============================================================================
# LOAD CORPUS
# ============================================================================

corpus = load_corpus(CORPUS_DIR)

print(f"\n✓ Corpus Loaded: {len(corpus)} documents\n")

# Build filename index for fast lookup
filename_index: Dict[str, LoreDocument] = {doc.filename: doc for doc in corpus}

# ============================================================================
# GROUND-TRUTH VALIDATION
# ============================================================================

# Moved from D-23 Cell 04: Ground-Truth Query Set (Aligned with D-21/D-22)
@dataclass
class GroundTruthQuery:
    """A single ground-truth query with expected relevant documents.

    Attributes:
        query_id: Unique query identifier (e.g., "Q-01")
        query_text: Natural language query
        query_type: One of SINGLE_HOP, MULTI_HOP, AUTHORITY, TEMPORAL, EXPLORATORY
        expected_filenames: List of corpus filenames expected to be relevant
        relevance_scores: Dict mapping filename → relevance grade (1-3)
    """
    query_id: str
    query_text: str
    query_type: str
    expected_filenames: List[str]
    relevance_scores: Dict[str, int]

# ============================================================================
# FULL QUERY SET: 36 queries (identical to D-21/D-22)
# ============================================================================

# SINGLE_HOP QUERIES (11 total)
# These test direct lookups of facts, entities, and concepts

QUERIES = [
    {
        "query_id": "Q-01",
        "type": "SINGLE_HOP",
        "text": "What is the Echo-Cant communication system?",
        "relevant_docs": ["000-codex_echo-cant.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-02",
        "type": "SINGLE_HOP",
        "text": "What are the defining characteristics of the Void-Marked?",
        "relevant_docs": ["db02-wb_void-marked-assembled-entry.md", "db03-dc_jotun-reader-chronology.md"],
    },
    {
        "query_id": "Q-03",
        "type": "SINGLE_HOP",
        "text": "Describe the Harrow-Sick condition and its effects.",
        "relevant_docs": ["000-codex_harrow-sick.md", "db02-wb_medical-phenomena-entry.md"],
    },
    {
        "query_id": "Q-04",
        "type": "SINGLE_HOP",
        "text": "What is the ODIN Protocol?",
        "relevant_docs": ["000-codex_odin-protocol.md", "standalone_aether-weave-os.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-05",
        "type": "SINGLE_HOP",
        "text": "What is the \u00c6ther-Weave operating system?",
        "relevant_docs": ["standalone_aether-weave-os.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-06",
        "type": "SINGLE_HOP",
        "text": "Who are the Scavenger-Barons and what do they do?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "db03-dc_contract-dispute-case-study.md"],
    },
    {
        "query_id": "Q-07",
        "type": "SINGLE_HOP",
        "text": "What is the Nine-Tiers architectural framework?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-27",
        "type": "SINGLE_HOP",
        "text": "What are the primary functions of the Warden-Host?",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-28",
        "type": "SINGLE_HOP",
        "text": "Define the Glitch in the context of Aethelgard's history.",
        "relevant_docs": ["000-resources_comprehensive-glossary.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-30",
        "type": "SINGLE_HOP",
        "text": "What is the Spell-Lock system?",
        "relevant_docs": ["000-codex_spell-lock.md", "standalone_aether-weave-os.md"],
    },
    {
        "query_id": "Q-34",
        "type": "SINGLE_HOP",
        "text": "Describe the Weir-Bone material and its properties.",
        "relevant_docs": ["db02-wb_weir-bone-assembled-entry.md", "000-resources_comprehensive-glossary.md"],
    },

    # MULTI_HOP QUERIES (8 total)
    # These require traversing relationships between multiple entities/concepts

    {
        "query_id": "Q-08",
        "type": "MULTI_HOP",
        "text": "How do Iron-Bane and God-Sleeper theological positions on Svin-fylking differ?",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md", "db02-wb_svin-fylking-assembled-entry.md"],
    },
    {
        "query_id": "Q-09",
        "type": "MULTI_HOP",
        "text": "What is the relationship between the Harrow-Sick and the Void-Marked?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db02-wb_void-marked-assembled-entry.md", "db03-dc_medical-causality-research.md"],
    },
    {
        "query_id": "Q-10",
        "type": "MULTI_HOP",
        "text": "How do the Scavenger-Barons use Spell-Lock in their operations?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "000-codex_spell-lock.md", "db03-dc_contract-dispute-case-study.md"],
    },
    {
        "query_id": "Q-11",
        "type": "MULTI_HOP",
        "text": "Explain the conflict between Warden-Host sanctuary protocols and Iron-Bane territorial claims.",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db03-dc_iron-bane-theological-analysis.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-12",
        "type": "MULTI_HOP",
        "text": "How does the ODIN Protocol interact with the \u00c6ther-Weave OS?",
        "relevant_docs": ["000-codex_odin-protocol.md", "standalone_aether-weave-os.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-29",
        "type": "MULTI_HOP",
        "text": "What role does the Echo-Cant system play in maintaining Warden-Host operations?",
        "relevant_docs": ["000-codex_echo-cant.md", "db02-wb_warden-host-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-31",
        "type": "MULTI_HOP",
        "text": "How do Void-Marked and Weir-Bone materials interact in salvage contexts?",
        "relevant_docs": ["db02-wb_void-marked-assembled-entry.md", "db02-wb_weir-bone-assembled-entry.md", "db03-dc_salvage-operations-manual.md"],
    },
    {
        "query_id": "Q-35",
        "type": "MULTI_HOP",
        "text": "What is the relationship between the Nine-Tiers architecture and Svin-fylking religious doctrine?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "db02-wb_svin-fylking-assembled-entry.md", "db03-dc_god-sleeper-operational-doctrine.md"],
    },

    # AUTHORITY QUERIES (5 total)
    # These test knowledge of canonical vs. draft vs. superseded information

    {
        "query_id": "Q-13",
        "type": "AUTHORITY",
        "text": "What is the canonical explanation for how the Glitch occurred?",
        "relevant_docs": ["000-resources_comprehensive-glossary.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-14",
        "type": "AUTHORITY",
        "text": "What are the established facts about the God-Sleeper movement origins?",
        "relevant_docs": ["db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-15",
        "type": "AUTHORITY",
        "text": "Which interpretations of Harrow-Sick etiology are considered canon?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db03-dc_medical-causality-research.md"],
    },
    {
        "query_id": "Q-16",
        "type": "AUTHORITY",
        "text": "What is the official Warden-Host stance on Void-Marked rights?",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db02-wb_void-marked-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-17",
        "type": "AUTHORITY",
        "text": "According to published sources, what materials constitute a valid Spell-Lock?",
        "relevant_docs": ["000-codex_spell-lock.md", "000-resources_comprehensive-glossary.md"],
    },

    # TEMPORAL QUERIES (6 total)
    # These test time-based historical retrieval across the Aethelgard timeline

    {
        "query_id": "Q-18",
        "type": "TEMPORAL",
        "text": "What major events happened in the first century after the Glitch (Year 0-100 PG)?",
        "relevant_docs": ["db03-dc_jotun-reader-chronology.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-19",
        "type": "TEMPORAL",
        "text": "When was the Warden-Host sanctuary established and what precipitated it?",
        "relevant_docs": ["db03-dc_sanctuary-establishment-record.md", "db02-wb_warden-host-assembled-entry.md"],
    },
    {
        "query_id": "Q-20",
        "type": "TEMPORAL",
        "text": "Trace the chronological development of Iron-Bane theological doctrine.",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-21",
        "type": "TEMPORAL",
        "text": "What is the timeline of major salvage discoveries in the Aethelgard region?",
        "relevant_docs": ["db03-dc_salvage-operations-manual.md", "db03-dc_jotun-reader-chronology.md"],
    },
    {
        "query_id": "Q-22",
        "type": "TEMPORAL",
        "text": "When did the God-Sleeper movement gain significant political influence?",
        "relevant_docs": ["db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-23",
        "type": "TEMPORAL",
        "text": "Describe the sequence of events in the Scavenger-Baron contract dispute.",
        "relevant_docs": ["db03-dc_contract-dispute-case-study.md", "db02-wb_scavenger-barons-assembled-entry.md"],
    },

    # EXPLORATORY QUERIES (6 total)
    # These test open-ended discovery of related concepts and themes

    {
        "query_id": "Q-24",
        "type": "EXPLORATORY",
        "text": "What are the major political tensions in post-Glitch Aethelgard?",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_contract-dispute-case-study.md", "db02-wb_scavenger-barons-assembled-entry.md"],
    },
    {
        "query_id": "Q-25",
        "type": "EXPLORATORY",
        "text": "How do different factions view the technological salvage efforts?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "db03-dc_salvage-operations-manual.md", "db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md"],
    },
    {
        "query_id": "Q-26",
        "type": "EXPLORATORY",
        "text": "What medical and physiological mysteries remain unsolved in Aethelgard?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db03-dc_medical-causality-research.md", "db02-wb_void-marked-assembled-entry.md"],
    },
    {
        "query_id": "Q-32",
        "type": "EXPLORATORY",
        "text": "What are the intersections between religious doctrine and technological systems in Aethelgard?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "db02-wb_svin-fylking-assembled-entry.md", "000-codex_odin-protocol.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-33",
        "type": "EXPLORATORY",
        "text": "How do material properties (Weir-Bone, Void-Marked) influence cultural practices?",
        "relevant_docs": ["db02-wb_weir-bone-assembled-entry.md", "db02-wb_void-marked-assembled-entry.md", "db02-wb_warden-host-assembled-entry.md"],
    },
    {
        "query_id": "Q-36",
        "type": "EXPLORATORY",
        "text": "What gaps exist in the documented understanding of Aethelgard's pre-Glitch history?",
        "relevant_docs": ["db03-dc_historical-theological-survey.md", "db03-dc_jotun-reader-chronology.md", "000-resources_comprehensive-glossary.md"],
    },
]

# ============================================================================
# BUILD GROUND_TRUTH_QUERIES from QUERIES (for compatibility with Cell 14)
# ============================================================================

GROUND_TRUTH_QUERIES: List[GroundTruthQuery] = []
for q in QUERIES:
    # Build uniform relevance scores (all docs score 2 by default)
    relevance = {doc: 2 for doc in q["relevant_docs"]}
    GROUND_TRUTH_QUERIES.append(GroundTruthQuery(
        query_id=q["query_id"],
        query_text=q["text"],
        query_type=q["type"],
        expected_filenames=q["relevant_docs"],
        relevance_scores=relevance,
    ))

# ============================================================================
# VALIDATION
# ============================================================================

query_type_counts = {}
for q in QUERIES:
    query_type_counts[q["type"]] = query_type_counts.get(q["type"], 0) + 1

assert len(QUERIES) == 36, f"Expected 36 queries, got {len(QUERIES)}"
assert query_type_counts.get("SINGLE_HOP", 0) == 11, f"Expected 11 SINGLE_HOP"
assert query_type_counts.get("MULTI_HOP", 0) == 8, f"Expected 8 MULTI_HOP"
assert query_type_counts.get("AUTHORITY", 0) == 5, f"Expected 5 AUTHORITY"
assert query_type_counts.get("TEMPORAL", 0) == 6, f"Expected 6 TEMPORAL"
assert query_type_counts.get("EXPLORATORY", 0) == 6, f"Expected 6 EXPLORATORY"

print(f"✓ All 36 ground-truth queries validated (aligned with D-21/D-22)")
print(f"  Distribution: {query_type_counts}")
print(f"  GROUND_TRUTH_QUERIES: {len(GROUND_TRUTH_QUERIES)} GroundTruthQuery objects")

# ============================================================================
# GROUND-TRUTH VALIDATION
# ============================================================================

print("Ground-Truth Validation:")
all_expected = set()
for q in GROUND_TRUTH_QUERIES:
    all_expected.update(q.expected_filenames)

missing = all_expected - set(filename_index.keys())
found = all_expected & set(filename_index.keys())

if missing:
    print(f"  ☢ {len(missing)} expected files NOT in corpus:")
    for fn in sorted(missing):
        print(f"      - {fn}")
else:
    print(f"  ✓ All {len(found)} expected files found in corpus")

# ============================================================================
# CORPUS SUMMARY
# ============================================================================

print(f"\nCorpus Summary:")
total_body_tokens = sum(estimate_tokens(doc.body) for doc in corpus)
print(f"  Documents:            {len(corpus)}")
print(f"  Total body tokens:    {total_body_tokens:,}")
print(f"  Avg tokens/doc:       {total_body_tokens // max(len(corpus), 1):,}")

# Metadata coverage
meta_keys = set()
for doc in corpus:
    meta_keys.update(doc.metadata.keys())
print(f"  Unique metadata keys: {len(meta_keys)}")

# Key field coverage
for field in ["type", "name", "canon", "era", "related_entities"]:
    present = sum(1 for doc in corpus if doc.metadata.get(field))
    pct = 100 * present / max(len(corpus), 1)
    print(f"  {field:20s}: {present}/{len(corpus)} ({pct:.0f}%)")

print("\n✓ Corpus ready for chunking.")


## Methodology: Multi-Layer Enrichment & GO/NO-GO Decision

### Research Hypothesis

**H1 (Primary)**: Multi-layer enrichment (D-23) significantly outperforms single-layer enrichment (D-22) in at least 2 of 4 retrieval metrics.

**H2 (Secondary)**: Entity and Relational layers provide the largest marginal improvement beyond the single-layer prefix.

**H3 (Tertiary)**: Authority-sensitive queries (Q-01 to Q-12) and temporal queries (Q-13 to Q-24) benefit more from multi-layer enrichment than factual queries (Q-25 to Q-36).

### The 8 Context Layers

Each chunk receives a multi-layer prefix constructed from these layers in order:

| # | Layer | Varies By | Token Budget | Source Field |
|---|-------|-----------|-------------|--------------|
| 1 | Corpus | Constant | ~5 | Hardcoded: CORPUS_LABEL |
| 2 | Domain | Document | ~8 | metadata.type → DOMAIN_CATEGORY_MAP |
| 3 | Entity | Document | ~8 | metadata.name |
| 4 | Authority | Document | ~8 | metadata.canon → authority mapping |
| 5 | Temporal | Document | ~12 | metadata.era (list → "X and Y") |
| 6 | Relational | Document | ~15-30 | metadata.related_entities (parsed) |
| 7 | Section | **Chunk** | ~10 | chunk.section_heading |
| 8 | Content | **Chunk** | variable | chunk.text (raw content) |

**Key difference from D-22**: The Section layer varies per chunk (not per document), so every chunk gets a unique prefix. This is more granular than D-22's static document-level prefix.

### Token Budget Impact

| Model | Max Tokens | Prefix Reserve | Content Budget | Prefix % |
|-------|-----------|---------------|---------------|----------|
| v2-moe | 512 | 100 | 250 | ~29% |
| **v1.5** | 8192 | 150 | 450 | ~25% |
| bge-m3 | 8192 | 150 | 874 | ~15% |

D-23 chunks are **shorter** than D-21/D-22 to accommodate the multi-layer prefix within the model's context window.

### 3-Way Comparison Design

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 08: Hybrid Chunking Engine (Round 4 — D-21 Algorithm Port)

CRITICAL FIX (Round 4):
  Rounds 1-3 used a reimplemented chunker that omitted:
    1. min_chunk_tokens=128 merge logic → produced 1,233 tiny chunks
    2. #{1,6} heading regex (used #{2,4} instead)

  This cell now uses D-21 Cell 8's EXACT algorithm.
  The ONLY modification: prefix_reserve subtracted from token budget.

Ported from D-21 Cell 8:
  - split_into_sections(): #{1,6} heading regex
  - sliding_window_split(): overlapping windows for oversized sections
  - chunk_document(): min_chunk_tokens merge logic preserved
"""

import re
from typing import List, Tuple, Optional, Dict, Any

# ============================================================================
# CONSTANTS — Match D-21 exactly
# ============================================================================

MIN_CHUNK_TOKENS = 128  # D-21: config.min_chunk_tokens = 128
OVERLAP_TOKENS = 150    # D-21: config.overlap_tokens = 150

# ============================================================================
# DATA STRUCTURES
# ============================================================================

@dataclass
class Chunk:
    """A single text chunk with metadata."""
    chunk_id: str
    doc_filename: str
    section_heading: Optional[str]
    text: str
    token_count_approx: int
    metadata: Dict[str, Any]


# ============================================================================
# TOKEN ESTIMATION — Identical to D-21
# ============================================================================

def estimate_tokens(text: str) -> int:
    """Approximate token count using word_count * 1.3 heuristic.
    Identical to D-21 Cell 8."""
    if not text or not text.strip():
        return 0
    return max(1, int(len(text.split()) * 1.3))


# ============================================================================
# HEADING EXTRACTION — D-21's exact regex and logic
# ============================================================================

def split_into_sections(body: str) -> List[Tuple[str, str]]:
    """Split a markdown body into (heading, content) pairs.

    PORTED FROM D-21 CELL 8 — EXACT COPY.
    Splits on lines starting with # through ###### (levels 1-6).
    The first section may have heading="" if text precedes the first heading.
    """
    # D-21's exact regex: matches ALL heading levels 1-6
    pattern = r'^(#{1,6}\s+.+)$'
    parts = re.split(pattern, body, flags=re.MULTILINE)

    sections = []
    current_heading = ""
    current_content = ""

    for part in parts:
        part_stripped = part.strip()
        if re.match(r'^#{1,6}\s+', part_stripped):
            # Save previous section if it has content
            if current_content.strip():
                sections.append((current_heading, current_content.strip()))
            current_heading = part_stripped
            current_content = ""
        else:
            current_content += part

    # Don't forget the last section
    if current_content.strip():
        sections.append((current_heading, current_content.strip()))

    # If no sections found, treat entire body as one section
    if not sections and body.strip():
        sections = [("", body.strip())]

    return sections


# ============================================================================
# SLIDING WINDOW — D-21's exact logic
# ============================================================================

def sliding_window_split(text: str, max_tokens: int, overlap_tokens: int) -> List[str]:
    """Split text into overlapping windows when it exceeds max_tokens.

    PORTED FROM D-21 CELL 8 — EXACT COPY.
    Uses word-level splitting with token estimation.
    """
    words = text.split()
    # Convert token limits to approximate word counts
    max_words = int(max_tokens / 1.3)
    overlap_words = int(overlap_tokens / 1.3)
    step = max(1, max_words - overlap_words)

    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i + max_words]
        chunks.append(" ".join(chunk_words))
        i += step
        if i + max_words >= len(words) and i < len(words):
            # Last chunk: take remaining words
            chunks.append(" ".join(words[i:]))
            break

    return chunks


# ============================================================================
# DOCUMENT CHUNKING — D-21's exact algorithm + prefix_reserve
# ============================================================================

def chunk_document(
    doc: LoreDocument,
    max_chunk_tokens: int,
    prefix_reserve: Optional[int] = None,
) -> List[Chunk]:
    """Chunk a document using D-21's hybrid heading + sliding-window approach.

    PORTED FROM D-21 CELL 8 with ONE modification:
      prefix_reserve is subtracted from the token budget so the
      enrichment prefix fits within max_chunk_tokens.

    D-21 behavior preserved:
      - split_into_sections() with #{1,6} heading regex
      - MIN_CHUNK_TOKENS=128 merge logic for short sections
      - sliding_window_split() for oversized sections
      - OVERLAP_TOKENS=150
    """
    if prefix_reserve is None:
        prefix_reserve = MODEL_CONFIG.prefix_reserve_tokens

    # The ONLY difference from D-21: subtract prefix_reserve from budget
    # D-21 effective_max = config.max_chunk_tokens (1024, full budget)
    # D-23 effective_max = max_chunk_tokens - prefix_reserve (1024 - 100 = 924)
    effective_max = max_chunk_tokens - prefix_reserve

    chunks = []
    chunk_counter = 0

    # Step 1: Split body into sections by markdown headings
    # D-21's split_into_sections: #{1,6} regex
    sections = split_into_sections(doc.body)

    for heading, content in sections:
        token_est = estimate_tokens(content)

        if token_est <= effective_max:
            # Section fits within limit — check if it meets minimum
            if token_est >= MIN_CHUNK_TOKENS:
                # Normal-sized section — create single chunk
                chunk_counter += 1
                chunks.append(Chunk(
                    chunk_id=f"{doc.filename}#chunk_{chunk_counter:03d}",
                    doc_filename=doc.filename,
                    section_heading=heading if heading else None,
                    text=content,
                    token_count_approx=token_est,
                    metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
                ))
            else:
                # *** D-21's MERGE LOGIC — was missing in D-23 Rounds 1-3 ***
                # Section below MIN_CHUNK_TOKENS — merge with previous chunk
                if chunks:
                    prev = chunks[-1]
                    merged_text = prev.text + "\n\n" + content
                    merged_tokens = estimate_tokens(merged_text)
                    if merged_tokens <= effective_max:
                        # Merge into previous chunk
                        chunks[-1] = Chunk(
                            chunk_id=prev.chunk_id,
                            doc_filename=prev.doc_filename,
                            section_heading=prev.section_heading,
                            text=merged_text,
                            token_count_approx=merged_tokens,
                            metadata=prev.metadata.copy() if hasattr(prev.metadata, 'copy') else dict(prev.metadata),
                        )
                    else:
                        # Can't merge (would exceed budget) — keep as small chunk
                        chunk_counter += 1
                        chunks.append(Chunk(
                            chunk_id=f"{doc.filename}#chunk_{chunk_counter:03d}",
                            doc_filename=doc.filename,
                            section_heading=heading if heading else None,
                            text=content,
                            token_count_approx=token_est,
                            metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
                        ))
                else:
                    # First chunk and it's small — keep it anyway
                    chunk_counter += 1
                    chunks.append(Chunk(
                        chunk_id=f"{doc.filename}#chunk_{chunk_counter:03d}",
                        doc_filename=doc.filename,
                        section_heading=heading if heading else None,
                        text=content,
                        token_count_approx=token_est,
                        metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
                    ))
        else:
            # Section too large — apply sliding window (D-21's logic)
            sub_texts = sliding_window_split(
                content,
                effective_max,
                OVERLAP_TOKENS,
            )
            for sub_text in sub_texts:
                chunk_counter += 1
                chunks.append(Chunk(
                    chunk_id=f"{doc.filename}#chunk_{chunk_counter:03d}",
                    doc_filename=doc.filename,
                    section_heading=heading if heading else None,
                    text=sub_text,
                    token_count_approx=estimate_tokens(sub_text),
                    metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
                ))

    # Edge case: document with no body at all (from D-21)
    if not chunks and doc.body.strip():
        chunks.append(Chunk(
            chunk_id=f"{doc.filename}#chunk_001",
            doc_filename=doc.filename,
            section_heading=None,
            text=doc.body.strip(),
            token_count_approx=estimate_tokens(doc.body),
            metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
        ))

    return chunks


# ============================================================================
# CONFIGURATION SUMMARY
# ============================================================================

effective = MODEL_CONFIG.max_chunk_tokens - MODEL_CONFIG.prefix_reserve_tokens
print("=" * 70)
print("CHUNKING ENGINE — D-21 Algorithm Port (Round 4 Fix)")
print("=" * 70)
print(f"\n  Model:                {MODEL_CONFIG.name}")
print(f"  Max chunk tokens:     {MODEL_CONFIG.max_chunk_tokens}")
print(f"  Prefix reserve:       {MODEL_CONFIG.prefix_reserve_tokens} tokens")
print(f"  Effective for content:{effective} tokens")
print(f"  Min chunk tokens:     {MIN_CHUNK_TOKENS}  [D-21 match]")
print(f"  Overlap:              {OVERLAP_TOKENS} tokens  [D-21 match]")
print(f"  Heading regex:        #{{1,6}}  [D-21 match]")
print(f"  Token estimator:      word_count x 1.3  [D-21 match]")
print(f"  Merge logic:          ENABLED  [D-21 match — was MISSING in R1-R3]")
print(f"\n  [Round 4] D-21's exact chunking algorithm ported.")
print(f"  Only change: prefix_reserve ({MODEL_CONFIG.prefix_reserve_tokens} tokens) subtracted from budget.")
print(f"  Expected chunk count: ~200-250 (was 1,233 without merge logic)")
print("=" * 70)

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 09: Ablation-Aware Enrichment Builder

Replaces D-23's hardcoded 7-layer build_multi_layer_prefix() with a
config-driven build_ablation_prefix() that activates only the layers
specified by the current ablation configuration.

Key changes from D-23 Cell 09:
  1. ABLATION_CONFIGS dict defines 5 layer combinations
  2. build_ablation_prefix() accepts a config_name parameter
  3. Only builds layers listed in the active config
  4. Individual layer builder functions unchanged from D-23

Layer builders reused from D-23:
  - build_domain_layer(metadata)
  - build_entity_layer(metadata)
  - build_authority_layer(metadata)
  - build_relational_layer(metadata)
  - build_section_layer(section_heading)

Dropped builders:
  - build_corpus_layer() — constant, non-discriminative
  - build_temporal_layer() — 0% population
"""

# ============================================================================
# ABLATION CONFIGURATIONS
# ============================================================================

ABLATION_CONFIGS = {
    "raw":              [],
    "domain_entity":    ["Domain", "Entity"],
    "de_authority":     ["Domain", "Entity", "Authority"],
    "de_section":       ["Domain", "Entity", "Section"],
    "de_relationships": ["Domain", "Entity", "Relationships"],
}

ABLATION_CONFIG_ORDER = ["raw", "domain_entity", "de_authority", "de_section", "de_relationships"]

# Prefix reserve per config: raw gets 0 (match D-21), others get 100
ABLATION_PREFIX_RESERVE = {
    "raw": 0,
    "domain_entity": 100,
    "de_authority": 100,
    "de_section": 100,
    "de_relationships": 100,
}

print("=" * 70)
print("D-24 ABLATION CONFIGURATIONS")
print("=" * 70)
for cfg in ABLATION_CONFIG_ORDER:
    layers = ABLATION_CONFIGS[cfg]
    reserve = ABLATION_PREFIX_RESERVE[cfg]
    layer_str = " + ".join(layers) if layers else "(none)"
    print(f"  {cfg:20s}: {layer_str:45s} reserve={reserve}")
print("=" * 70)

# ============================================================================
# DOMAIN CATEGORY MAPPING (from D-23)
# ============================================================================

DOMAIN_CATEGORY_MAP: Dict[str, str] = {
    "faction":    "organizations",
    "character":  "individuals",
    "location":   "geography",
    "event":      "history",
    "item":       "artifacts",
    "concept":    "metaphysics",
    "spell":      "magic system",
    "creature":   "bestiary",
    "region":     "geography",
    "timeline":   "chronology",
}


# ============================================================================
# LAYER BUILDER FUNCTIONS (unchanged from D-23)
# ============================================================================

def build_domain_layer(metadata: Dict[str, Any]) -> str:
    doc_type = str(metadata.get("type", "unknown")).lower().strip()
    category = DOMAIN_CATEGORY_MAP.get(doc_type, "general")
    return f"Domain: This content is from a {doc_type} document in the {category} category."


def build_entity_layer(metadata: Dict[str, Any]) -> Optional[str]:
    name = str(metadata.get("name", "")).strip()
    if not name or name.lower() == "unknown":
        return None
    return f"Entity: This content describes {name}."


def build_authority_layer(metadata: Dict[str, Any]) -> str:
    canon = metadata.get("canon", "")
    if isinstance(canon, bool):
        canon_str = "true" if canon else "false"
    else:
        canon_str = str(canon).lower().strip()

    if canon_str in ("true", "yes", "canonical"):
        authority_text = "canonical and authoritative"
    elif canon_str == "apocryphal":
        authority_text = "apocryphal (non-canonical, speculative)"
    elif canon_str == "deprecated":
        authority_text = "deprecated and superseded"
    else:
        authority_text = "draft (not yet canonical)"

    return f"Authority: This content is {authority_text}."


def build_relational_layer(metadata: Dict[str, Any]) -> Optional[str]:
    rel_parts = []

    # Standard relationships
    relationships = metadata.get("relationships", [])
    if isinstance(relationships, list):
        for rel in relationships:
            if isinstance(rel, dict):
                target = rel.get("target", "")
                if isinstance(target, str):
                    target_name = target.split("/")[-1].replace(".md", "").replace("-", " ").title()
                else:
                    target_name = str(target)
                rel_type = str(rel.get("type", "related to")).replace("_", " ")
                rel_parts.append(f"{rel_type} {target_name}")

    # Cross-references (added in D-23 R3)
    cross_refs = metadata.get("cross_references", [])
    if isinstance(cross_refs, list):
        for ref in cross_refs:
            if isinstance(ref, str):
                ref_name = ref.split("/")[-1].replace(".md", "").replace("-", " ").title()
                rel_parts.append(f"related to {ref_name}")

    # Factions
    factions = metadata.get("factions", [])
    if isinstance(factions, list):
        for faction in factions:
            if isinstance(faction, str):
                rel_parts.append(f"associated with {faction}")

    # Locations
    location = metadata.get("location", metadata.get("region", ""))
    if isinstance(location, str) and location.strip():
        rel_parts.append(f"located in {location.strip()}")
    elif isinstance(location, list):
        for loc in location:
            if isinstance(loc, str) and loc.strip():
                rel_parts.append(f"located in {loc.strip()}")

    if not rel_parts:
        return None

    return f"Relationships: {'; '.join(rel_parts)}."


def build_section_layer(section_heading: Optional[str]) -> Optional[str]:
    if not section_heading or not section_heading.strip():
        return None
    heading_clean = section_heading.strip().lstrip("#").strip()
    if not heading_clean:
        return None
    return f"Section: This content is from the {heading_clean} section."


# ============================================================================
# ABLATION PREFIX BUILDER (replaces build_multi_layer_prefix)
# ============================================================================

# Map layer names to builder functions
LAYER_BUILDER_MAP = {
    "Domain":        lambda meta, _: build_domain_layer(meta),
    "Entity":        lambda meta, _: build_entity_layer(meta),
    "Authority":     lambda meta, _: build_authority_layer(meta),
    "Relationships": lambda meta, _: build_relational_layer(meta),
    "Section":       lambda _, sh: build_section_layer(sh),
}


def build_ablation_prefix(
    config_name: str,
    metadata: Dict[str, Any],
    section_heading: Optional[str] = None,
) -> Tuple[str, Dict[str, int]]:
    """Build a prefix using only the layers active in the given config.

    Args:
        config_name: Key into ABLATION_CONFIGS (e.g., "domain_entity")
        metadata: Normalized document metadata dict
        section_heading: Section heading for the Section layer

    Returns:
        Tuple of (prefix_text, layer_token_audit)
    """
    active_layers = ABLATION_CONFIGS[config_name]
    layer_token_audit: Dict[str, int] = {}
    prefix_parts: List[str] = []

    for layer_name in active_layers:
        builder_fn = LAYER_BUILDER_MAP[layer_name]
        layer_text = builder_fn(metadata, section_heading)
        if layer_text is not None:
            prefix_parts.append(layer_text)
            layer_token_audit[layer_name] = estimate_tokens(layer_text)

    prefix_text = "\n\n".join(prefix_parts)
    return prefix_text, layer_token_audit


def build_ablation_chunk(
    config_name: str,
    chunk: Chunk,
) -> Tuple[Chunk, Dict[str, int]]:
    """Build an enriched chunk using the ablation config.

    Args:
        config_name: Key into ABLATION_CONFIGS
        chunk: Original Chunk object (raw text)

    Returns:
        Tuple of (enriched_chunk, layer_token_audit)
    """
    prefix_text, layer_audit = build_ablation_prefix(
        config_name, chunk.metadata, chunk.section_heading
    )

    if prefix_text:
        enriched_text = prefix_text + "\n\n" + chunk.text
    else:
        enriched_text = chunk.text

    enriched_chunk = Chunk(
        chunk_id=chunk.chunk_id,
        doc_filename=chunk.doc_filename,
        section_heading=chunk.section_heading,
        text=enriched_text,
        token_count_approx=estimate_tokens(enriched_text),
        metadata=chunk.metadata.copy() if hasattr(chunk.metadata, 'copy') else dict(chunk.metadata),
    )

    return enriched_chunk, layer_audit


print("\n✓ Ablation enrichment builder ready")
print(f"  Configs: {len(ABLATION_CONFIGS)}")
print(f"  Layer builders: {list(LAYER_BUILDER_MAP.keys())}")

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 10: Multi-Config Chunk, Enrich & Audit

Runs the chunk → enrich → audit pipeline once per ablation configuration.
Each config gets its own enriched_chunks list and audit data.

Key difference from D-23 Cell 10:
  - Outer loop over ABLATION_CONFIG_ORDER
  - Config A (raw) uses prefix_reserve=0 for D-21-equivalent chunking
  - Stores results in ablation_results dict keyed by config name

Outputs:
  - ablation_results: Dict[str, Dict] with per-config data:
      - "enriched_chunks": List[Chunk]
      - "layer_audits": List[Dict]
      - "tokens_before": List[int]
      - "tokens_after": List[int]
      - "overflow_count": int
"""

ablation_results: Dict[str, Dict] = {}

print("=" * 80)
print("D-24 MULTI-CONFIG CHUNK, ENRICH & AUDIT")
print("=" * 80)

for config_name in ABLATION_CONFIG_ORDER:
    prefix_reserve = ABLATION_PREFIX_RESERVE[config_name]
    active_layers = ABLATION_CONFIGS[config_name]
    layer_str = " + ".join(active_layers) if active_layers else "(none — raw)"

    print(f"\n{'─' * 70}")
    print(f"CONFIG: {config_name}")
    print(f"  Layers: {layer_str}")
    print(f"  Prefix reserve: {prefix_reserve} tokens")
    print(f"{'─' * 70}")

    enriched_chunks: List[Chunk] = []
    layer_audits: List[Dict[str, Any]] = []
    tokens_before: List[int] = []
    tokens_after: List[int] = []
    overflow_count = 0

    for doc in tqdm(corpus, desc=f"  {config_name}"):
        # Chunk with config-specific prefix reserve
        doc_chunks = chunk_document(doc, MODEL_CONFIG.max_chunk_tokens, prefix_reserve=prefix_reserve)

        for chunk in doc_chunks:
            tokens_before.append(chunk.token_count_approx)

            enriched_chunk, layer_audit = build_ablation_chunk(config_name, chunk)
            tokens_after.append(enriched_chunk.token_count_approx)

            if enriched_chunk.token_count_approx > MODEL_CONFIG.max_chunk_tokens:
                overflow_count += 1

            enriched_chunks.append(enriched_chunk)

            audit_record = {
                "config": config_name,
                "chunk_id": chunk.chunk_id,
                "doc_filename": chunk.doc_filename,
                "section_heading": chunk.section_heading or "(none)",
                "tokens_raw": chunk.token_count_approx,
                "tokens_enriched": enriched_chunk.token_count_approx,
                **layer_audit,
            }
            layer_audits.append(audit_record)

    # Store results
    ablation_results[config_name] = {
        "enriched_chunks": enriched_chunks,
        "layer_audits": layer_audits,
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "overflow_count": overflow_count,
    }

    # Summary for this config
    arr_before = np.array(tokens_before)
    arr_after = np.array(tokens_after)
    arr_overhead = arr_after - arr_before

    print(f"\n  ✓ {len(enriched_chunks)} chunks")
    print(f"  Overflow: {overflow_count}/{len(enriched_chunks)} ({100*overflow_count/max(len(enriched_chunks),1):.1f}%)")
    print(f"  Mean tokens: {arr_before.mean():.0f} raw → {arr_after.mean():.0f} enriched (overhead: {arr_overhead.mean():.0f})")

# ============================================================================
# CROSS-CONFIG SUMMARY
# ============================================================================

print(f"\n{'=' * 80}")
print("CROSS-CONFIG SUMMARY")
print(f"{'=' * 80}")
print(f"\n  {'Config':20s} {'Chunks':>7s} {'Overflow':>10s} {'Mean Raw':>10s} {'Mean Enr':>10s} {'Overhead':>10s}")
print(f"  {'─'*20} {'─'*7} {'─'*10} {'─'*10} {'─'*10} {'─'*10}")

for cfg in ABLATION_CONFIG_ORDER:
    r = ablation_results[cfg]
    n = len(r["enriched_chunks"])
    ovf = r["overflow_count"]
    mean_raw = np.mean(r["tokens_before"])
    mean_enr = np.mean(r["tokens_after"])
    overhead = mean_enr - mean_raw
    print(f"  {cfg:20s} {n:>7d} {ovf:>4d}/{n:<5d} {mean_raw:>10.0f} {mean_enr:>10.0f} {overhead:>+10.0f}")

# Export unified token audit
all_audits = []
for cfg in ABLATION_CONFIG_ORDER:
    all_audits.extend(ablation_results[cfg]["layer_audits"])

audit_df = pd.DataFrame(all_audits)
audit_csv_path = OUTPUT_DIR / "d24_layer_token_audits.csv"
audit_df.to_csv(audit_csv_path, index=False)
print(f"\n✓ Token audit exported: {audit_csv_path} ({len(audit_df)} rows)")

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 11: Per-Config Embedding & ChromaDB Indexing

Creates one ChromaDB collection per ablation config. Embeds all enriched
chunks and indexes them using the same pipeline as D-23 Cell 11.

Collections: d24_raw, d24_domain_entity, d24_de_authority, d24_de_section, d24_de_relationships
"""

import chromadb

client = chromadb.Client()
ablation_collections: Dict[str, Any] = {}

print("=" * 80)
print("D-24 PER-CONFIG EMBEDDING & INDEXING")
print("=" * 80)

for config_name in ABLATION_CONFIG_ORDER:
    enriched_chunks = ablation_results[config_name]["enriched_chunks"]

    collection_name = f"d24_{config_name}"
    collection = client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    print(f"\n  [{config_name}] Embedding {len(enriched_chunks)} chunks...")

    # Batch encode
    texts = [c.text for c in enriched_chunks]
    embeddings = model.encode(texts, prompt_name="search_document", batch_size=64, show_progress_bar=True)

    # Index in ChromaDB
    ids = [c.chunk_id for c in enriched_chunks]
    metadatas = [{"doc_filename": c.doc_filename, "section": c.section_heading or ""} for c in enriched_chunks]

    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=metadatas,
    )

    ablation_collections[config_name] = collection
    print(f"  ✓ {collection_name}: {collection.count()} vectors indexed")

print(f"\n{'=' * 80}")
print(f"✓ {len(ablation_collections)} collections created")
for name, col in ablation_collections.items():
    print(f"  {name:20s}: {col.count()} vectors")
print(f"{'=' * 80}")

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 12: Per-Config Query Execution

Runs all 36 ground-truth queries against each ablation config's collection.
Stores results in ablation_query_results dict.
"""

ablation_query_results: Dict[str, Dict[str, Dict]] = {}

print("=" * 80)
print("D-24 PER-CONFIG QUERY EXECUTION")
print("=" * 80)

for config_name in ABLATION_CONFIG_ORDER:
    collection = ablation_collections[config_name]
    config_results: Dict[str, Dict] = {}

    print(f"\n  [{config_name}] Querying {len(GROUND_TRUTH)} queries against {collection.count()} vectors...")

    for query_id, query_info in GROUND_TRUTH.items():
        query_text = query_info["query"]

        # Encode query
        query_embedding = model.encode(query_text, prompt_name="search_query")

        # Retrieve top-10
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=10,
            include=["distances", "documents", "metadatas"],
        )

        # Extract document filenames from chunk IDs
        retrieved_ids = results["ids"][0]
        retrieved_docs = [rid.split("#")[0] for rid in retrieved_ids]
        distances = results["distances"][0]

        config_results[query_id] = {
            "query_type": query_info["type"],
            "retrieved_ids": retrieved_ids,
            "retrieved_docs": retrieved_docs,
            "distances": distances,
            "relevant_docs": query_info["relevant"],
        }

    ablation_query_results[config_name] = config_results
    print(f"  ✓ {config_name}: {len(config_results)} queries executed")

print(f"\n✓ All queries complete across {len(ablation_query_results)} configs")

## D-24 Results: Ablation Comparison Analysis

All 36 ground-truth queries have been executed against each of the 5 ablation
configurations. The following cells compute metrics, deltas, statistical
significance, and visualizations to determine which layers contribute most
to retrieval improvement.

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 14: Metric Computation Functions

Defines the four retrieval evaluation metrics used across D-21, D-22, and D-23:
  1. Precision@K — fraction of top-K results that are relevant
  2. Recall@K — fraction of all relevant documents found in top-K
  3. NDCG@K — ranking quality using graded relevance with log discount
  4. MRR — reciprocal rank of first relevant result

Also defines compute_all_metrics() which runs all four on a list of QueryResults.

These functions are identical to D-21 Cell 14 and D-22 Cell 14.
"""


def precision_at_k(
    retrieved_ids: List[str],
    relevant_ids: set,
    k: int = K_PRECISION,
) -> float:
    """Compute Precision@K.

    Formula: P@K = |relevant ∩ top-K| / K

    Args:
        retrieved_ids: Ordered list of retrieved chunk IDs (rank order)
        relevant_ids: Set of known-relevant chunk IDs
        k: Cutoff rank (default K_PRECISION=5)

    Returns:
        Precision@K in [0.0, 1.0]

    Example:
        >>> precision_at_k(["A","B","C","D","E"], {"A","C","F"}, k=5)
        0.4  # 2 relevant in top-5
    """
    if k <= 0:
        return 0.0
    top_k = set(retrieved_ids[:k])
    return len(top_k & set(relevant_ids)) / k


def recall_at_k(
    retrieved_ids: List[str],
    relevant_ids: set,
    k: int = K_RECALL,
) -> float:
    """Compute Recall@K.

    Formula: R@K = |relevant ∩ top-K| / |relevant|

    Args:
        retrieved_ids: Ordered list of retrieved chunk IDs
        relevant_ids: Set of known-relevant chunk IDs
        k: Cutoff rank (default K_RECALL=10)

    Returns:
        Recall@K in [0.0, 1.0]. Returns 0.0 if relevant set is empty.

    Example:
        >>> recall_at_k(["A","B","C"], {"A","C","D","E"}, k=3)
        0.5  # 2 of 4 relevant found
    """
    if not relevant_ids:
        return 0.0
    top_k = set(retrieved_ids[:k])
    return len(top_k & set(relevant_ids)) / len(relevant_ids)


def ndcg_at_k(
    retrieved_ids: List[str],
    relevance_scores: Dict[str, int],
    k: int = K_NDCG,
) -> float:
    """Compute NDCG@K (Normalized Discounted Cumulative Gain).

    Formula:
      DCG@K  = Σ(i=1..K) rel(i) / log2(i + 1)
      IDCG@K = DCG of ideal ranking (sorted by relevance descending)
      NDCG@K = DCG@K / IDCG@K

    Uses graded relevance scores (1=marginal, 2=relevant, 3=highly relevant).

    IMPORTANT: Retrieved IDs are deduplicated before scoring. In chunk-level
    retrieval with document-level relevance, multiple chunks from the same
    document map to the same ID after .split("#")[0]. Without dedup, DCG
    accumulates gains for every duplicate while IDCG only counts unique docs,
    producing NDCG > 1.0 (mathematically impossible).
    [FIX: Round 3 — dedup retrieved IDs before DCG computation]

    Args:
        retrieved_ids: Ordered list of retrieved chunk/document IDs
        relevance_scores: Dict mapping doc_id → relevance grade
        k: Cutoff rank (default K_NDCG=10)

    Returns:
        NDCG@K in [0.0, 1.0]. Returns 0.0 if no relevant documents exist.

    Example:
        >>> ndcg_at_k(["A","B","C"], {"A": 3, "C": 1}, k=3)
        # DCG = 3/log2(2) + 0/log2(3) + 1/log2(4) = 3.0 + 0 + 0.5 = 3.5
        # IDCG = 3/log2(2) + 1/log2(3) = 3.0 + 0.63 = 3.63
        # NDCG = 3.5 / 3.63 ≈ 0.964
    """
    if not relevance_scores:
        return 0.0

    # Deduplicate: only keep first occurrence of each document ID.
    # This prevents counting the same document multiple times when
    # multiple chunks from one doc appear in the top-k results.
    seen = set()
    deduped = []
    for rid in retrieved_ids[:k]:
        if rid not in seen:
            seen.add(rid)
            deduped.append(rid)

    # Compute DCG on deduplicated list
    dcg = 0.0
    for i, chunk_id in enumerate(deduped):
        rel = relevance_scores.get(chunk_id, 0)
        dcg += rel / np.log2(i + 2)  # i+2 because 0-indexed

    # Compute ideal DCG (IDCG)
    ideal_rels = sorted(relevance_scores.values(), reverse=True)[:k]
    idcg = 0.0
    for i, rel in enumerate(ideal_rels):
        idcg += rel / np.log2(i + 2)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def mean_reciprocal_rank(
    retrieved_ids: List[str],
    relevant_ids: set,
) -> float:
    """Compute Mean Reciprocal Rank (MRR).

    Formula: MRR = 1 / rank_of_first_relevant_result

    Args:
        retrieved_ids: Ordered list of retrieved chunk IDs
        relevant_ids: Set of known-relevant chunk IDs

    Returns:
        MRR in [0.0, 1.0]. Returns 0.0 if no relevant result found.

    Example:
        >>> mean_reciprocal_rank(["X","Y","A","B"], {"A","C"})
        0.333  # First relevant at rank 3
    """
    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def compute_all_metrics(
    qr_list: List[QueryResult],
    gt_queries: List[GroundTruthQuery],
) -> pd.DataFrame:
    """Compute all four metrics for each query.

    Matches QueryResult objects to GroundTruthQuery objects by query_id.
    Returns a DataFrame with one row per query.

    Args:
        qr_list: List of QueryResult objects from query execution
        gt_queries: List of GroundTruthQuery objects with relevance judgments

    Returns:
        DataFrame with columns:
          [query_id, query_type, precision@5, recall@10, ndcg@10, mrr]
    """
    # Build lookup: query_id → ground truth
    gt_lookup = {q.query_id: q for q in gt_queries}

    rows = []
    for qr in qr_list:
        gt = gt_lookup.get(qr.query_id)
        if gt is None:
            continue

        retrieved = [r["chunk_id"].split("#")[0] for r in qr.results]
        relevant = set(gt.expected_filenames)

        rows.append({
            "query_id": qr.query_id,
            "query_type": qr.query_type,
            "precision@5": precision_at_k(retrieved, relevant),
            "recall@10": recall_at_k(retrieved, relevant),
            "ndcg@10": ndcg_at_k(retrieved, gt.relevance_scores),
            "mrr": mean_reciprocal_rank(retrieved, relevant),
        })

    return pd.DataFrame(rows)


# ============================================================================
# VERIFICATION
# ============================================================================

print("✓ Metric Computation Functions Loaded")
print(f"  Functions: precision_at_k, recall_at_k, ndcg_at_k, mean_reciprocal_rank")
print(f"  compute_all_metrics → DataFrame[query_id, query_type, P@5, R@10, NDCG@10, MRR]")
print(f"  Evaluation constants: K_PRECISION={K_PRECISION}, K_RECALL={K_RECALL}, K_NDCG={K_NDCG}")
print("✓ Ready for metric computation.")


In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 15: Per-Config Metric Computation & Baseline Loading

Computes P@5, R@10, NDCG@10, MRR for each ablation config.
Loads D-21 and D-22 results for cross-experiment comparison.

Outputs:
  - ablation_metrics: Dict[config_name, pd.DataFrame] — per-query metrics
  - d21_results_df, d22_results_df — prior experiment data
"""

# ============================================================================
# COMPUTE D-24 METRICS PER CONFIG
# ============================================================================

ablation_metrics: Dict[str, pd.DataFrame] = {}

for config_name in ABLATION_CONFIG_ORDER:
    config_results = ablation_query_results[config_name]
    rows = []

    for query_id, qr in config_results.items():
        relevant = set(qr["relevant_docs"])
        retrieved = qr["retrieved_docs"]

        p5 = precision_at_k(relevant, retrieved, k=5)
        r10 = recall_at_k(relevant, retrieved, k=10)
        ndcg = ndcg_at_k(relevant, retrieved, k=10)
        mrr_val = mrr(relevant, retrieved)

        rows.append({
            "config": config_name,
            "query_id": query_id,
            "query_type": qr["query_type"],
            "precision@5": p5,
            "recall@10": r10,
            "ndcg@10": ndcg,
            "mrr": mrr_val,
        })

    df = pd.DataFrame(rows)
    ablation_metrics[config_name] = df
    means = df[["precision@5", "recall@10", "ndcg@10", "mrr"]].mean()
    print(f"  {config_name:20s}: P@5={means['precision@5']:.4f}  R@10={means['recall@10']:.4f}  "
          f"NDCG={means['ndcg@10']:.4f}  MRR={means['mrr']:.4f}")

# ============================================================================
# LOAD D-21 AND D-22 BASELINES
# ============================================================================

print(f"\nLoading prior experiment results...")

d21_results_path = Path("d21-output/d21_results.csv")
d22_results_path = Path("d22-output/d22_results.csv")

d21_available = d21_results_path.exists()
d22_available = d22_results_path.exists()

if d21_available:
    d21_results_df = pd.read_csv(d21_results_path)
    print(f"  ✓ D-21: {len(d21_results_df)} rows loaded")
else:
    print(f"  ✗ D-21 baseline not found at {d21_results_path}")
    d21_results_df = None

if d22_available:
    d22_results_df = pd.read_csv(d22_results_path)
    print(f"  ✓ D-22: {len(d22_results_df)} rows loaded")
else:
    print(f"  ✗ D-22 single-layer not found at {d22_results_path}")
    d22_results_df = None

# ============================================================================
# SUMMARY TABLE
# ============================================================================

print(f"\n{'=' * 80}")
print("OVERALL METRICS SUMMARY")
print(f"{'=' * 80}")
print(f"\n  {'Config':20s} {'P@5':>8s} {'R@10':>8s} {'NDCG@10':>8s} {'MRR':>8s}")
print(f"  {'─'*20} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")

if d21_available:
    d21_v15 = d21_results_df[d21_results_df["model"] == "v1.5"]
    d21_means = d21_v15[["precision@5", "recall@10", "ndcg@10", "mrr"]].mean()
    print(f"  {'D-21 (baseline)':20s} {d21_means['precision@5']:>8.4f} {d21_means['recall@10']:>8.4f} "
          f"{d21_means['ndcg@10']:>8.4f} {d21_means['mrr']:>8.4f}")

if d22_available:
    d22_means = d22_results_df[["precision@5", "recall@10", "ndcg@10", "mrr"]].mean()
    print(f"  {'D-22 (single-layer)':20s} {d22_means['precision@5']:>8.4f} {d22_means['recall@10']:>8.4f} "
          f"{d22_means['ndcg@10']:>8.4f} {d22_means['mrr']:>8.4f}")

print(f"  {'─'*20} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")

for config_name in ABLATION_CONFIG_ORDER:
    df = ablation_metrics[config_name]
    means = df[["precision@5", "recall@10", "ndcg@10", "mrr"]].mean()
    print(f"  {config_name:20s} {means['precision@5']:>8.4f} {means['recall@10']:>8.4f} "
          f"{means['ndcg@10']:>8.4f} {means['mrr']:>8.4f}")

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 16: Ablation Delta Analysis

Computes per-query metric deltas for:
  1. Each config vs D-21 baseline
  2. Each config vs D-22 single-layer
  3. Each config vs Config A (raw — internal control)
  4. Marginal contribution of each added layer (C-B, D-B, E-B)

Outputs:
  - delta_vs_d21: Dict[config, pd.DataFrame]
  - delta_vs_d22: Dict[config, pd.DataFrame]
  - delta_vs_raw: Dict[config, pd.DataFrame]
  - marginal_contributions: pd.DataFrame
"""

METRICS = ["precision@5", "recall@10", "ndcg@10", "mrr"]

# ============================================================================
# DELTAS VS D-21
# ============================================================================

delta_vs_d21: Dict[str, pd.DataFrame] = {}

if d21_available:
    d21_v15 = d21_results_df[d21_results_df["model"] == "v1.5"].copy()
    print("DELTAS VS D-21 (BASELINE)")
    print("-" * 60)

    for config_name in ABLATION_CONFIG_ORDER:
        cfg_df = ablation_metrics[config_name]

        merged = cfg_df.merge(d21_v15, on="query_id", suffixes=("_d24", "_d21"))
        for m in METRICS:
            merged[f"delta_{m}"] = merged[f"{m}_d24"] - merged[f"{m}_d21"]

        delta_vs_d21[config_name] = merged
        means = merged[[f"delta_{m}" for m in METRICS]].mean()
        print(f"  {config_name:20s}: " + "  ".join(f"Δ{m.split('@')[0]}={means[f'delta_{m}']:+.4f}" for m in METRICS))

# ============================================================================
# DELTAS VS D-22
# ============================================================================

delta_vs_d22: Dict[str, pd.DataFrame] = {}

if d22_available:
    print(f"\nDELTAS VS D-22 (SINGLE-LAYER)")
    print("-" * 60)

    for config_name in ABLATION_CONFIG_ORDER:
        cfg_df = ablation_metrics[config_name]

        merged = cfg_df.merge(d22_results_df, on="query_id", suffixes=("_d24", "_d22"))
        for m in METRICS:
            merged[f"delta_{m}"] = merged[f"{m}_d24"] - merged[f"{m}_d22"]

        delta_vs_d22[config_name] = merged
        means = merged[[f"delta_{m}" for m in METRICS]].mean()
        print(f"  {config_name:20s}: " + "  ".join(f"Δ{m.split('@')[0]}={means[f'delta_{m}']:+.4f}" for m in METRICS))

# ============================================================================
# DELTAS VS RAW (INTERNAL CONTROL)
# ============================================================================

delta_vs_raw: Dict[str, pd.DataFrame] = {}
raw_df = ablation_metrics["raw"]

print(f"\nDELTAS VS RAW (INTERNAL CONTROL)")
print("-" * 60)

for config_name in ABLATION_CONFIG_ORDER:
    if config_name == "raw":
        continue

    cfg_df = ablation_metrics[config_name]
    merged = cfg_df.merge(raw_df, on="query_id", suffixes=("_cfg", "_raw"))
    for m in METRICS:
        merged[f"delta_{m}"] = merged[f"{m}_cfg"] - merged[f"{m}_raw"]

    delta_vs_raw[config_name] = merged
    means = merged[[f"delta_{m}" for m in METRICS]].mean()
    print(f"  {config_name:20s}: " + "  ".join(f"Δ{m.split('@')[0]}={means[f'delta_{m}']:+.4f}" for m in METRICS))

# ============================================================================
# MARGINAL LAYER CONTRIBUTIONS
# ============================================================================

print(f"\nMARGINAL LAYER CONTRIBUTIONS (vs domain_entity base)")
print("-" * 60)

base_df = ablation_metrics["domain_entity"]
marginal_rows = []

layer_configs = {
    "Authority":     "de_authority",
    "Section":       "de_section",
    "Relationships": "de_relationships",
}

for layer_name, config_name in layer_configs.items():
    cfg_df = ablation_metrics[config_name]
    merged = cfg_df.merge(base_df, on="query_id", suffixes=("_cfg", "_base"))

    row = {"layer": layer_name}
    for m in METRICS:
        delta = merged[f"{m}_cfg"].mean() - merged[f"{m}_base"].mean()
        row[f"delta_{m}"] = delta
    marginal_rows.append(row)
    print(f"  +{layer_name:15s}: " + "  ".join(f"Δ{m.split('@')[0]}={row[f'delta_{m}']:+.4f}" for m in METRICS))

marginal_contributions = pd.DataFrame(marginal_rows)

# Export deltas
for config_name, df in delta_vs_d21.items():
    csv_path = OUTPUT_DIR / f"d24_delta_{config_name}_vs_d21.csv"
    df.to_csv(csv_path, index=False)

for config_name, df in delta_vs_d22.items():
    csv_path = OUTPUT_DIR / f"d24_delta_{config_name}_vs_d22.csv"
    df.to_csv(csv_path, index=False)

marginal_csv = OUTPUT_DIR / "d24_marginal_contributions.csv"
marginal_contributions.to_csv(marginal_csv, index=False)
print(f"\n✓ Deltas and marginal contributions exported to {OUTPUT_DIR}")

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 17: Statistical Significance — Per-Config Wilcoxon Tests

Performs Wilcoxon signed-rank tests for:
  1. Each config vs D-21 baseline (if available)
  2. Each config vs D-22 single-layer (if available)
  3. Each enriched config vs raw (internal control)

Bonferroni correction: α = 0.05 / 3 comparisons = 0.0167
"""

from scipy.stats import wilcoxon

ALPHA = 0.05
N_COMPARISONS = 3
BONFERRONI_ALPHA = ALPHA / N_COMPARISONS

sig_rows = []

def run_wilcoxon(label, metric, values_a, values_b):
    """Run Wilcoxon signed-rank test and return result dict."""
    diff = values_a - values_b
    nonzero = np.sum(diff != 0)

    if nonzero < 5:
        return {
            "pair_label": label, "metric": metric,
            "statistic": None, "pvalue": None,
            "significant": False, "effect_size": None,
            "effect_magnitude": "insufficient_data", "n_nonzero": int(nonzero),
        }

    stat, pval = wilcoxon(values_a, values_b, alternative="two-sided")
    r = 1 - (2 * stat) / (nonzero * (nonzero + 1))

    if abs(r) >= 0.5:
        magnitude = "large"
    elif abs(r) >= 0.3:
        magnitude = "medium"
    else:
        magnitude = "small"

    return {
        "pair_label": label, "metric": metric,
        "statistic": float(stat), "pvalue": float(pval),
        "significant": pval < BONFERRONI_ALPHA, "effect_size": float(r),
        "effect_magnitude": magnitude, "n_nonzero": int(nonzero),
    }

print("=" * 80)
print(f"STATISTICAL SIGNIFICANCE (Bonferroni α = {BONFERRONI_ALPHA:.4f})")
print("=" * 80)

# vs D-21
if d21_available:
    d21_v15 = d21_results_df[d21_results_df["model"] == "v1.5"]
    for config_name in ABLATION_CONFIG_ORDER:
        cfg_df = ablation_metrics[config_name]
        merged = cfg_df.merge(d21_v15, on="query_id", suffixes=("_d24", "_d21"))
        label = f"D-24 {config_name} vs D-21"
        for m in METRICS:
            result = run_wilcoxon(label, m, merged[f"{m}_d24"].values, merged[f"{m}_d21"].values)
            sig_rows.append(result)

# vs D-22
if d22_available:
    for config_name in ABLATION_CONFIG_ORDER:
        cfg_df = ablation_metrics[config_name]
        merged = cfg_df.merge(d22_results_df, on="query_id", suffixes=("_d24", "_d22"))
        label = f"D-24 {config_name} vs D-22"
        for m in METRICS:
            result = run_wilcoxon(label, m, merged[f"{m}_d24"].values, merged[f"{m}_d22"].values)
            sig_rows.append(result)

# vs raw (internal)
raw_df = ablation_metrics["raw"]
for config_name in ABLATION_CONFIG_ORDER:
    if config_name == "raw":
        continue
    cfg_df = ablation_metrics[config_name]
    merged = cfg_df.merge(raw_df, on="query_id", suffixes=("_cfg", "_raw"))
    label = f"D-24 {config_name} vs raw"
    for m in METRICS:
        result = run_wilcoxon(label, m, merged[f"{m}_cfg"].values, merged[f"{m}_raw"].values)
        sig_rows.append(result)

sig_df = pd.DataFrame(sig_rows)
sig_csv = OUTPUT_DIR / "d24_significance_results.csv"
sig_df.to_csv(sig_csv, index=False)

# Print significant results
sig_only = sig_df[sig_df["significant"] == True]
print(f"\n  {len(sig_only)} significant results (of {len(sig_df)} tests):")
for _, row in sig_only.iterrows():
    print(f"    {row['pair_label']:40s} {row['metric']:12s} p={row['pvalue']:.6f} r={row['effect_size']:.3f} ({row['effect_magnitude']})")

print(f"\n✓ Significance results exported: {sig_csv}")

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 18: Ablation Visualization — Grouped Bar Chart

Creates a 2x2 figure comparing all 5 configs + D-21 + D-22 baselines
across the 4 metrics.
"""

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("D-24 Layer Ablation — Metric Comparison", fontsize=16, fontweight="bold")

METRICS_DISPLAY = {
    "precision@5": "Precision@5",
    "recall@10": "Recall@10",
    "ndcg@10": "NDCG@10",
    "mrr": "MRR",
}

# Collect means for all experiments
all_labels = []
all_means = {m: [] for m in METRICS}

# D-21 baseline
if d21_available:
    d21_v15 = d21_results_df[d21_results_df["model"] == "v1.5"]
    d21_means = d21_v15[METRICS].mean()
    all_labels.append("D-21\n(baseline)")
    for m in METRICS:
        all_means[m].append(d21_means[m])

# D-22 single-layer
if d22_available:
    d22_means = d22_results_df[METRICS].mean()
    all_labels.append("D-22\n(single)")
    for m in METRICS:
        all_means[m].append(d22_means[m])

# D-24 configs
config_labels = {
    "raw": "A: raw",
    "domain_entity": "B: D+E",
    "de_authority": "C: D+E+A",
    "de_section": "D: D+E+S",
    "de_relationships": "E: D+E+R",
}

for config_name in ABLATION_CONFIG_ORDER:
    df = ablation_metrics[config_name]
    means = df[METRICS].mean()
    all_labels.append(config_labels[config_name])
    for m in METRICS:
        all_means[m].append(means[m])

# Plot
colors = ["#888888", "#4CAF50"] + ["#2196F3", "#FF9800", "#E91E63", "#9C27B0", "#00BCD4"]
n_bars = len(all_labels)

for idx, (metric, display_name) in enumerate(METRICS_DISPLAY.items()):
    ax = axes[idx // 2][idx % 2]
    x = range(n_bars)
    bars = ax.bar(x, all_means[metric], color=colors[:n_bars], alpha=0.85, edgecolor="white")

    # Value labels
    for bar, val in zip(bars, all_means[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_title(display_name, fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(all_labels, fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
viz_path = OUTPUT_DIR / "visualization_ablation_comparison.png"
plt.savefig(viz_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {viz_path}")

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 19: Marginal Contribution Chart & Config Heatmap

Figure 1: Marginal contribution of each layer (vs domain_entity base)
Figure 2: Per-query-type performance heatmap across configs
"""

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

# ============================================================================
# FIGURE 1: MARGINAL CONTRIBUTIONS
# ============================================================================

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle("Marginal Layer Contribution (vs Domain+Entity base)", fontsize=14, fontweight="bold")

layers = marginal_contributions["layer"].tolist()
x = range(len(layers))
width = 0.2
metric_colors = {"precision@5": "#2196F3", "recall@10": "#4CAF50", "ndcg@10": "#FF9800", "mrr": "#E91E63"}

for i, (m, color) in enumerate(metric_colors.items()):
    vals = marginal_contributions[f"delta_{m}"].tolist()
    bars = ax.bar([xi + i*width for xi in x], vals, width, label=m, color=color, alpha=0.8)
    for bar, val in zip(bars, vals):
        y_pos = bar.get_height() + 0.002 if val >= 0 else bar.get_height() - 0.015
        ax.text(bar.get_x() + bar.get_width()/2, y_pos, f"{val:+.3f}", ha="center", fontsize=8)

ax.set_xticks([xi + 1.5*width for xi in x])
ax.set_xticklabels([f"+{l}" for l in layers], fontsize=11, fontweight="bold")
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_ylabel("Delta vs Domain+Entity")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
viz1_path = OUTPUT_DIR / "visualization_marginal_contributions.png"
plt.savefig(viz1_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {viz1_path}")

# ============================================================================
# FIGURE 2: PER-QUERY-TYPE HEATMAP
# ============================================================================

import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("D-24 Per-Query-Type Performance by Config", fontsize=14, fontweight="bold")

for idx, metric in enumerate(METRICS):
    ax = axes[idx // 2][idx % 2]

    # Build heatmap data: rows=query_types, cols=configs
    query_types = sorted(ablation_metrics["raw"]["query_type"].unique())
    heatmap_data = []

    for qt in query_types:
        row = []
        for cfg in ABLATION_CONFIG_ORDER:
            df = ablation_metrics[cfg]
            mean_val = df[df["query_type"] == qt][metric].mean()
            row.append(mean_val)
        heatmap_data.append(row)

    heatmap_df = pd.DataFrame(heatmap_data, index=query_types, columns=ABLATION_CONFIG_ORDER)
    sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="RdYlGn", ax=ax,
                vmin=0, vmax=1, linewidths=0.5, cbar_kws={"shrink": 0.8})
    ax.set_title(METRICS_DISPLAY.get(metric, metric), fontsize=12, fontweight="bold")

plt.tight_layout()
viz2_path = OUTPUT_DIR / "visualization_querytype_heatmap.png"
plt.savefig(viz2_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {viz2_path}")

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 20: Per-Config GO/NO-GO Scorecard

Evaluates 7 criteria for each ablation config (same criteria as D-23).
Identifies the best-performing configuration.
"""

print("=" * 80)
print("D-24 PER-CONFIG GO/NO-GO SCORECARD")
print("=" * 80)

config_scores: Dict[str, Dict] = {}

for config_name in ABLATION_CONFIG_ORDER:
    r = ablation_results[config_name]
    cfg_df = ablation_metrics[config_name]
    cfg_means = cfg_df[METRICS].mean()

    criteria = {}

    # [1] EXECUTION: overflow < 5%
    overflow_pct = 100 * r["overflow_count"] / max(len(r["enriched_chunks"]), 1)
    criteria["execution"] = overflow_pct < 5

    # [2] IMPROVEMENT over D-21: ≥3/4 metrics positive
    if d21_available:
        d21_v15 = d21_results_df[d21_results_df["model"] == "v1.5"]
        d21_means = d21_v15[METRICS].mean()
        positive_d21 = sum(1 for m in METRICS if cfg_means[m] > d21_means[m])
        criteria["improve_d21"] = positive_d21 >= 3
    else:
        criteria["improve_d21"] = None

    # [3] IMPROVEMENT over D-22: ≥2/4 metrics positive
    if d22_available:
        d22_means = d22_results_df[METRICS].mean()
        positive_d22 = sum(1 for m in METRICS if cfg_means[m] > d22_means[m])
        criteria["improve_d22"] = positive_d22 >= 2
    else:
        criteria["improve_d22"] = None

    # [4] Significance: ≥2 metrics significant vs D-21
    sig_vs_d21 = sig_df[sig_df["pair_label"].str.contains(f"D-24 {config_name} vs D-21")]
    n_sig_d21 = sig_vs_d21["significant"].sum() if len(sig_vs_d21) > 0 else 0
    criteria["significance"] = n_sig_d21 >= 2

    # [5] Marginal value: ≥1 metric >5% gain over D-22
    if d22_available:
        gains = [(cfg_means[m] - d22_means[m]) / max(d22_means[m], 0.001) for m in METRICS]
        criteria["marginal_value"] = any(g > 0.05 for g in gains)
    else:
        criteria["marginal_value"] = None

    # [6] No catastrophic degradation: <25% queries degraded vs D-21
    if config_name in delta_vs_d21:
        delta_df = delta_vs_d21[config_name]
        total_pairs = len(delta_df) * len(METRICS)
        degraded = sum(
            (delta_df[f"delta_{m}"] < 0).sum() for m in METRICS
        )
        degraded_pct = 100 * degraded / max(total_pairs, 1)
        criteria["no_catastrophic"] = degraded_pct < 25
    else:
        criteria["no_catastrophic"] = None

    # [7] Authority/Temporal benefit
    authority_mean = cfg_df[cfg_df["query_type"] == "AUTHORITY"]["mrr"].mean() if "AUTHORITY" in cfg_df["query_type"].values else 0
    factual_mean = cfg_df[cfg_df["query_type"] == "SINGLE_HOP"]["mrr"].mean() if "SINGLE_HOP" in cfg_df["query_type"].values else 0
    temporal_mean = cfg_df[cfg_df["query_type"] == "TEMPORAL"]["mrr"].mean() if "TEMPORAL" in cfg_df["query_type"].values else 0
    criteria["auth_temp_benefit"] = (authority_mean > factual_mean) or (temporal_mean > factual_mean)

    score = sum(1 for v in criteria.values() if v is True)
    total = sum(1 for v in criteria.values() if v is not None)

    config_scores[config_name] = {"criteria": criteria, "score": score, "total": total}

    # Print scorecard
    status_map = {True: "PASS ✅", False: "FAIL ❌", None: "N/A  ⚪"}
    print(f"\n  {config_name} — {score}/{total}")
    for k, v in criteria.items():
        print(f"    [{status_map[v]}] {k}")

# ============================================================================
# BEST CONFIG
# ============================================================================

best_config = max(config_scores.keys(), key=lambda k: config_scores[k]["score"])
best_score = config_scores[best_config]

print(f"\n{'=' * 80}")
print(f"BEST CONFIG: {best_config} ({best_score['score']}/{best_score['total']} criteria)")
print(f"{'=' * 80}")

# Export go/no-go
with open(OUTPUT_DIR / "d24_go_nogo_decision.txt", "w") as f:
    f.write("D-24 GO/NO-GO DECISION REPORT\n")
    f.write("=" * 80 + "\n")
    f.write(f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n")
    f.write(f"Bonferroni-adjusted α: {BONFERRONI_ALPHA:.4f}\n\n")
    for config_name in ABLATION_CONFIG_ORDER:
        cs = config_scores[config_name]
        f.write(f"{config_name}: {cs['score']}/{cs['total']} criteria passed\n")
        for k, v in cs["criteria"].items():
            status = "PASS" if v is True else ("FAIL" if v is False else "N/A")
            f.write(f"  [{status}] {k}\n")
        f.write("\n")
    f.write(f"BEST CONFIG: {best_config}\n")

print(f"✓ GO/NO-GO report exported")

In [ ]:
#!/usr/bin/env python3
"""
D-24 Cell 21: Export Results — Final Artifact Assembly

Exports all D-24 output artifacts.
"""

print("=" * 80)
print("D-24 EXPORT — FINAL ARTIFACTS")
print("=" * 80)

# Per-config results CSVs
for config_name in ABLATION_CONFIG_ORDER:
    df = ablation_metrics[config_name]
    csv_path = OUTPUT_DIR / f"d24_results_{config_name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"  ✓ {csv_path} ({len(df)} rows)")

# Summary CSV (all configs)
all_results = pd.concat([ablation_metrics[cfg] for cfg in ABLATION_CONFIG_ORDER], ignore_index=True)
summary_path = OUTPUT_DIR / "d24_results_all.csv"
all_results.to_csv(summary_path, index=False)
print(f"  ✓ {summary_path} ({len(all_results)} rows)")

# Marginal contributions
print(f"  ✓ d24_marginal_contributions.csv (already exported)")

# Significance
print(f"  ✓ d24_significance_results.csv (already exported)")

# Token audits
print(f"  ✓ d24_layer_token_audits.csv (already exported)")

# GO/NO-GO
print(f"  ✓ d24_go_nogo_decision.txt (already exported)")

# Visualizations
print(f"  ✓ visualization_ablation_comparison.png")
print(f"  ✓ visualization_marginal_contributions.png")
print(f"  ✓ visualization_querytype_heatmap.png")

# Manifest
print(f"\n{'=' * 80}")
print(f"COMPLETE MANIFEST ({OUTPUT_DIR}):")
print(f"{'=' * 80}")
import glob
for f in sorted(glob.glob(str(OUTPUT_DIR / "*"))):
    size = os.path.getsize(f)
    print(f"  {Path(f).name:50s} {size:>8,d} bytes")

## D-24 Ablation Summary

D-24 tested 5 enrichment configurations, from zero layers (raw) to 3 layers.
All configs used D-21's exact chunking algorithm with identical corpus and queries.

### Configurations Tested

| Config | Layers | Purpose |
|--------|--------|---------|
| A (raw) | None | Internal D-21 control |
| B (domain_entity) | Domain + Entity | Minimal enrichment |
| C (de_authority) | Domain + Entity + Authority | +canonical status |
| D (de_section) | Domain + Entity + Section | +document structure |
| E (de_relationships) | Domain + Entity + Relationships | +cross-references |

### Key Question Answered

**Which individual context layers contribute most to retrieval improvement?**

Review the marginal contribution chart and per-config GO/NO-GO scores above
to determine the optimal layer combination for production use.